In [6]:
import os, random, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models
import matplotlib.pyplot as plt

SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2
USE_IMAGENET_WEIGHTS = True
RESUME = False

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 GPU，请在 Kaggle 的 Notebook Settings -> Accelerator 中选择 GPU。"
    )

DEVICE = torch.device("cuda")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

# =========================
# 实验配置：X0 Baseline
# =========================
EXPERIMENT = "X0_Baseline"
USE_EDGE = False
EDGE_LAMBDA = 0.0

if EXPERIMENT == "X0_Baseline" and (USE_EDGE or EDGE_LAMBDA != 0.0):
    raise ValueError("X0 必须设置 USE_EDGE=False，EDGE_LAMBDA=0.0")

if EXPERIMENT == "X1_Edge_NoSupervision" and (
    not USE_EDGE or EDGE_LAMBDA != 0.0
):
    raise ValueError("X1 必须设置 USE_EDGE=True，EDGE_LAMBDA=0.0")

if EXPERIMENT == "X2_Edge_Supervision" and (
    not USE_EDGE or EDGE_LAMBDA <= 0.0
):
    raise ValueError("X2 必须设置 USE_EDGE=True，EDGE_LAMBDA>0.0")

OUT_ROOT = Path("/kaggle/working") / f"WHU_{EXPERIMENT}_final"
CKPT_DIR = OUT_ROOT / "checkpoints"
RESULT_DIR = OUT_ROOT / "results"
FIG_DIR = OUT_ROOT / "figures"

for p in (CKPT_DIR, RESULT_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT)
print("USE_EDGE:", USE_EDGE)
print("EDGE_LAMBDA:", EDGE_LAMBDA)
print("OUT_ROOT:", OUT_ROOT)

# =========================
# 自动寻找 WHU 数据
# =========================
MANUAL_WHU_ROOT = ""

def locate_whu():
    if MANUAL_WHU_ROOT.strip():
        root = Path(MANUAL_WHU_ROOT)
        candidates = [(
            root / "train" / "image",
            root / "train" / "label",
            root / "val" / "image",
            root / "val" / "label",
        )]
    else:
        candidates = []

        for top in (Path("/kaggle/input"), Path("/kaggle/working")):
            if not top.exists():
                continue

            for train_dir in top.rglob("train"):
                candidates.append((
                    train_dir / "image",
                    train_dir / "label",
                    train_dir.parent / "val" / "image",
                    train_dir.parent / "val" / "label",
                ))

    valid = [
        c for c in candidates
        if all(p.is_dir() for p in c)
    ]

    if not valid:
        raise FileNotFoundError(
            "没有找到 WHU 数据。请把 MANUAL_WHU_ROOT 改为包含 train 和 val 的目录。"
        )

    valid.sort(
        key=lambda c: (
            "whu" not in str(c[0]).lower(),
            len(str(c[0]))
        )
    )

    return valid[0]

TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL = locate_whu()

print("TRAIN_IMG:", TRAIN_IMG)
print("TRAIN_LBL:", TRAIN_LBL)
print("VAL_IMG  :", VAL_IMG)
print("VAL_LBL  :", VAL_LBL)

# =========================
# Dataset
# =========================
class WHUDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, img_size=256):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.img_size = img_size

        valid_ext = {
            ".png",
            ".jpg",
            ".jpeg",
            ".tif",
            ".tiff",
        }

        self.names = sorted(
            p.name
            for p in self.img_dir.iterdir()
            if p.is_file() and p.suffix.lower() in valid_ext
        )

        if not self.names:
            raise RuntimeError(f"没有找到图像：{self.img_dir}")

        missing = [
            name for name in self.names
            if not (self.lbl_dir / name).exists()
        ]

        if missing:
            raise FileNotFoundError(
                f"缺少同名标签，示例：{missing[:5]}"
            )

    def __len__(self):
        return len(self.names)

    @staticmethod
    def make_whu_edge(mask_np):
        mask_u8 = mask_np.astype(np.uint8) * 255
        kernel = np.ones((3, 3), np.uint8)
        dilated = cv2.dilate(
            mask_u8,
            kernel,
            iterations=1
        )

        return (
            (dilated - mask_u8) > 0
        ).astype(np.float32)

    def __getitem__(self, idx):
        name = self.names[idx]

        image = Image.open(
            self.img_dir / name
        ).convert("RGB")

        label = Image.open(
            self.lbl_dir / name
        ).convert("L")

        image = image.resize(
            (self.img_size, self.img_size),
            Image.BILINEAR
        )

        label = label.resize(
            (self.img_size, self.img_size),
            Image.NEAREST
        )

        image_tensor = T.ToTensor()(image)

        label_np = (
            np.array(label) > 127
        ).astype(np.uint8)

        edge_np = self.make_whu_edge(label_np)

        return (
            image_tensor,
            torch.from_numpy(label_np).long(),
            torch.from_numpy(edge_np).float(),
            name,
        )

train_ds = WHUDataset(
    TRAIN_IMG,
    TRAIN_LBL,
    IMG_SIZE
)

val_ds = WHUDataset(
    VAL_IMG,
    VAL_LBL,
    IMG_SIZE
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))

# =========================
# 模型
# =========================
class EdgeBranch(nn.Module):
    def __init__(self, in_channels=3, out_channels=32):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

        self.edge_head = nn.Conv2d(
            out_channels,
            1,
            kernel_size=1
        )

    def forward(self, x):
        feat = self.conv(x)
        edge_logits = self.edge_head(feat)
        return feat, edge_logits


class ResNetSemanticBranch(nn.Module):
    def __init__(self, out_channels=64):
        super().__init__()

        weights = (
            models.ResNet18_Weights.DEFAULT
            if USE_IMAGENET_WEIGHTS
            else None
        )

        try:
            resnet = models.resnet18(
                weights=weights
            )
        except Exception as exc:
            raise RuntimeError(
                "ResNet18 权重加载失败。"
                "请打开 Kaggle Internet，或者将 "
                "USE_IMAGENET_WEIGHTS=False。"
            ) from exc

        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        self.smooth = nn.Sequential(
            nn.Conv2d(
                512,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.smooth(feat)

        return F.interpolate(
            feat,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )


class EdgeEnhancedNet(nn.Module):
    def __init__(self, use_edge=True):
        super().__init__()

        self.use_edge = use_edge
        self.semantic_branch = ResNetSemanticBranch(64)

        if use_edge:
            self.edge_branch = EdgeBranch(3, 32)
            fusion_in = 96
        else:
            self.edge_branch = None
            fusion_in = 64

        self.fusion = nn.Sequential(
            nn.Conv2d(
                fusion_in,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                64,
                2,
                kernel_size=1
            ),
        )

    def forward(self, x):
        semantic_feat = self.semantic_branch(x)
        edge_logits = None

        if self.use_edge:
            edge_feat, edge_logits = self.edge_branch(x)
            fused = torch.cat(
                [semantic_feat, edge_feat],
                dim=1
            )
        else:
            fused = semantic_feat

        seg_logits = self.fusion(fused)

        return seg_logits, edge_logits


# =========================
# 损失函数
# =========================
ce_loss = nn.CrossEntropyLoss()
edge_bce = nn.BCEWithLogitsLoss()

def dice_loss_building(
    logits,
    target,
    eps=1e-6
):
    prob = torch.softmax(
        logits,
        dim=1
    )[:, 1]

    gt = (
        target == 1
    ).float()

    inter = (
        prob * gt
    ).sum(dim=(1, 2))

    denom = (
        prob.sum(dim=(1, 2))
        + gt.sum(dim=(1, 2))
    )

    dice = (
        2.0 * inter + eps
    ) / (
        denom + eps
    )

    return (
        1.0 - dice
    ).mean()


# =========================
# 评价指标
# =========================
def get_boundary(mask):
    mask_u8 = mask.astype(np.uint8) * 255
    kernel = np.ones((3, 3), np.uint8)

    dilated = cv2.dilate(
        mask_u8,
        kernel,
        iterations=1
    )

    eroded = cv2.erode(
        mask_u8,
        kernel,
        iterations=1
    )

    return (
        (dilated - eroded) > 0
    ).astype(np.uint8)


def boundary_metrics(
    gt,
    pred,
    tolerance=2
):
    gt_boundary = get_boundary(gt)
    pred_boundary = get_boundary(pred)

    intersection = np.logical_and(
        gt_boundary,
        pred_boundary
    ).sum()

    union = np.logical_or(
        gt_boundary,
        pred_boundary
    ).sum()

    boundary_iou = (
        intersection / (union + 1e-8)
    )

    kernel = np.ones(
        (
            2 * tolerance + 1,
            2 * tolerance + 1
        ),
        np.uint8
    )

    gt_dilated = cv2.dilate(
        gt_boundary,
        kernel
    )

    pred_dilated = cv2.dilate(
        pred_boundary,
        kernel
    )

    matched_pred = np.logical_and(
        pred_boundary == 1,
        gt_dilated == 1
    ).sum()

    matched_gt = np.logical_and(
        gt_boundary == 1,
        pred_dilated == 1
    ).sum()

    precision = matched_pred / (
        (pred_boundary == 1).sum()
        + 1e-8
    )

    recall = matched_gt / (
        (gt_boundary == 1).sum()
        + 1e-8
    )

    boundary_f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return float(boundary_f1), float(boundary_iou)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    boundary_f1_list = []
    boundary_iou_list = []

    for images, labels, _, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        seg_logits, _ = model(images)

        pred = torch.argmax(
            seg_logits,
            dim=1
        ).cpu().numpy().astype(np.uint8)

        gt = labels.numpy().astype(np.uint8)

        tp += np.logical_and(
            pred == 1,
            gt == 1
        ).sum()

        fp += np.logical_and(
            pred == 1,
            gt == 0
        ).sum()

        fn += np.logical_and(
            pred == 0,
            gt == 1
        ).sum()

        tn += np.logical_and(
            pred == 0,
            gt == 0
        ).sum()

        for p, g in zip(pred, gt):
            bf1, biou = boundary_metrics(
                g,
                p,
                tolerance=2
            )

            boundary_f1_list.append(bf1)
            boundary_iou_list.append(biou)

    building_iou = tp / (
        tp + fp + fn + 1e-8
    )

    background_iou = tn / (
        tn + fp + fn + 1e-8
    )

    precision = tp / (
        tp + fp + 1e-8
    )

    recall = tp / (
        tp + fn + 1e-8
    )

    f1 = 2 * precision * recall / (
        precision + recall + 1e-8
    )

    return {
        "BuildingIoU": float(building_iou),
        "True_mIoU": float(
            (building_iou + background_iou) / 2
        ),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "BoundaryF1": float(
            np.mean(boundary_f1_list)
        ),
        "BoundaryIoU": float(
            np.mean(boundary_iou_list)
        ),
    }


def count_params_m(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ) / 1e6


# =========================
# 训练
# =========================
seed_everything(SEED)

model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

last_path = CKPT_DIR / f"{EXPERIMENT}_last.pth"
best_path = CKPT_DIR / f"{EXPERIMENT}_best.pth"

best_iou = -1.0
history = []
start_epoch = 1

if RESUME and last_path.exists():
    checkpoint = torch.load(
        last_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

    best_iou = float(
        checkpoint.get(
            "best_iou",
            -1.0
        )
    )

    history = checkpoint.get(
        "history",
        []
    )

    start_epoch = int(
        checkpoint["epoch"]
    ) + 1

    print(
        f"从第 {start_epoch - 1} 轮恢复，"
        f"当前最佳 IoU={best_iou:.6f}"
    )

for epoch in range(
    start_epoch,
    EPOCHS + 1
):
    model.train()

    total_loss = 0.0
    total_seg_loss = 0.0
    total_edge_loss = 0.0
    batches = 0

    t0 = time.time()

    for images, labels, edges, _ in train_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        edges = edges.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            enabled=True
        ):
            seg_logits, edge_logits = model(images)

            loss_seg = (
                ce_loss(seg_logits, labels)
                + dice_loss_building(
                    seg_logits,
                    labels
                )
            )

            if USE_EDGE and EDGE_LAMBDA > 0.0:
                loss_edge = edge_bce(
                    edge_logits.squeeze(1),
                    edges
                )
            else:
                loss_edge = torch.zeros(
                    (),
                    device=DEVICE
                )

            loss = (
                loss_seg
                + EDGE_LAMBDA * loss_edge
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(
            loss.item()
        )

        total_seg_loss += float(
            loss_seg.item()
        )

        total_edge_loss += float(
            loss_edge.item()
        )

        batches += 1

    avg_loss = total_loss / max(
        batches,
        1
    )

    val_metrics = evaluate(
        model,
        val_loader
    )

    history.append({
        "epoch": epoch,
        "train_loss": avg_loss,
        "train_seg_loss": (
            total_seg_loss / max(batches, 1)
        ),
        "train_edge_loss": (
            total_edge_loss / max(batches, 1)
        ),
        **val_metrics,
    })

    if val_metrics["BuildingIoU"] > best_iou:
        best_iou = val_metrics["BuildingIoU"]

        torch.save(
            model.state_dict(),
            best_path
        )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_iou": best_iou,
            "history": history,
            "experiment": EXPERIMENT,
        },
        last_path
    )

    pd.DataFrame(history).to_csv(
        RESULT_DIR / f"{EXPERIMENT}_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss={avg_loss:.4f} | "
        f"IoU={val_metrics['BuildingIoU'] * 100:.2f}% | "
        f"F1={val_metrics['F1'] * 100:.2f}% | "
        f"BF1={val_metrics['BoundaryF1'] * 100:.2f}%"
    )

# =========================
# 最终评估
# =========================
if not best_path.exists():
    raise FileNotFoundError(
        f"没有找到最佳权重：{best_path}"
    )

best_model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

best_model.load_state_dict(
    torch.load(
        best_path,
        map_location=DEVICE
    )
)

metrics = evaluate(
    best_model,
    val_loader
)

metrics_pct = {
    k: v * 100.0
    for k, v in metrics.items()
}

config = {
    "experiment": EXPERIMENT,
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "lr": LR,
    "use_edge": USE_EDGE,
    "edge_lambda": EDGE_LAMBDA,
    "main_loss": "CrossEntropy + Dice",
    "edge_loss": (
        "BCEWithLogitsLoss"
        if EDGE_LAMBDA > 0
        else "disabled"
    ),
    "whu_edge_label": "3x3 Dilate(M)-M",
    "best_checkpoint_rule": (
        "maximum validation Building IoU"
    ),
    "postprocessing": "none",
    "imagenet_pretrained": USE_IMAGENET_WEIGHTS,
    "params_M": count_params_m(best_model),
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "train_img": str(TRAIN_IMG),
    "val_img": str(VAL_IMG),
    "device": torch.cuda.get_device_name(0),
}

(RESULT_DIR / f"{EXPERIMENT}_config.json").write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

(RESULT_DIR / f"{EXPERIMENT}_metrics.json").write_text(
    json.dumps(
        metrics_pct,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

report_lines = [
    f"WHU formal ablation experiment: {EXPERIMENT}",
    "=" * 70,
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    "",
    "Best checkpoint validation metrics (%)",
]

report_lines.extend(
    f"{k}: {v:.4f}"
    for k, v in metrics_pct.items()
)

report = "\n".join(report_lines)

(RESULT_DIR / f"{EXPERIMENT}_report.txt").write_text(
    report,
    encoding="utf-8"
)

# =========================
# 保存训练曲线
# =========================
hist = pd.DataFrame(history)

plt.figure(figsize=(7.2, 4.6))
plt.plot(
    hist["epoch"],
    hist["train_loss"],
    linewidth=1.8
)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title(f"{EXPERIMENT} Training Curve")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_training_loss.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

# =========================
# 保存指标图
# =========================
metric_names = [
    "BuildingIoU",
    "F1",
    "BoundaryF1",
    "BoundaryIoU",
]

metric_values = [
    metrics_pct[k]
    for k in metric_names
]

plt.figure(figsize=(7.2, 4.6))

bars = plt.bar(
    metric_names,
    metric_values
)

plt.ylim(0, 100)
plt.ylabel("Score (%)")
plt.title(f"{EXPERIMENT} Validation Metrics")

for bar, value in zip(
    bars,
    metric_values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_metrics.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

# =========================
# 打包结果
# =========================
archive_path = Path(
    "/kaggle/working"
) / f"{EXPERIMENT}_artifacts.zip"

with zipfile.ZipFile(
    archive_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for file in OUT_ROOT.rglob("*"):
        if file.is_file():
            zf.write(
                file,
                file.relative_to(
                    OUT_ROOT.parent
                )
            )

print("\n" + report)
print("\n结果压缩包：", archive_path)

Device: cuda
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
Experiment: X0_Baseline
USE_EDGE: False
EDGE_LAMBDA: 0.0
OUT_ROOT: /kaggle/working/WHU_X0_Baseline_final
TRAIN_IMG: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
TRAIN_LBL: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/label
VAL_IMG  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
VAL_LBL  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/label
Train samples: 4736
Val samples  : 1036
Epoch 01/50 | Loss=0.9084 | IoU=48.21% | F1=65.06% | BF1=28.21%
Epoch 02/50 | Loss=0.7809 | IoU=52.31% | F1=68.69% | BF1=31.07%
Epoch 03/50 | Loss=0.7285 | IoU=56.48% | F1=72.19% | BF1=37.09%
Epoch 04/50 | Loss=0.6856 | IoU=54.90% | F1=70.88% | BF1=37.02%
Epoch 05/50 | Loss=0.6648 | IoU=59.33% | F1=74.47% | BF1=40.08%
Epoch 06/50 | Loss=0.6568 | IoU=58.68% | F1=73.96% | BF1=38.81%
Epoch 07/50 | Loss=0.6188 | IoU=60.11% | F1=75.09% | BF1=40.38%
Epoch 08/50 | Loss=0.6055 | IoU=59.44% 

In [1]:
import os, random, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models
import matplotlib.pyplot as plt

# =========================
# 0. 固定实验参数
# =========================
SEED = 2026
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2

USE_IMAGENET_WEIGHTS = True
RESUME = False

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 GPU，请在 Kaggle Notebook Settings -> Accelerator 中选择 GPU。"
    )

DEVICE = torch.device("cuda")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

# =========================
# 1. X1 实验配置
# =========================
EXPERIMENT = "X1_Edge_NoSupervision_seed2026"

# X1：保留 EdgeBranch，关闭边缘监督
USE_EDGE = True
EDGE_LAMBDA = 0.0

if not USE_EDGE:
    raise ValueError("X1 必须设置 USE_EDGE=True")

if EDGE_LAMBDA != 0.0:
    raise ValueError("X1 必须设置 EDGE_LAMBDA=0.0")

OUT_ROOT = Path("/kaggle/working") / f"WHU_{EXPERIMENT}_final"
CKPT_DIR = OUT_ROOT / "checkpoints"
RESULT_DIR = OUT_ROOT / "results"
FIG_DIR = OUT_ROOT / "figures"

for p in (CKPT_DIR, RESULT_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT)
print("USE_EDGE:", USE_EDGE)
print("EDGE_LAMBDA:", EDGE_LAMBDA)
print("OUT_ROOT:", OUT_ROOT)

# =========================
# 2. 自动寻找 WHU 数据
# =========================
MANUAL_WHU_ROOT = ""

def locate_whu():
    if MANUAL_WHU_ROOT.strip():
        root = Path(MANUAL_WHU_ROOT)

        candidates = [(
            root / "train" / "image",
            root / "train" / "label",
            root / "val" / "image",
            root / "val" / "label",
        )]
    else:
        candidates = []

        for top in (
            Path("/kaggle/input"),
            Path("/kaggle/working")
        ):
            if not top.exists():
                continue

            for train_dir in top.rglob("train"):
                candidates.append((
                    train_dir / "image",
                    train_dir / "label",
                    train_dir.parent / "val" / "image",
                    train_dir.parent / "val" / "label",
                ))

    valid = [
        c for c in candidates
        if all(p.is_dir() for p in c)
    ]

    if not valid:
        raise FileNotFoundError(
            "没有找到 WHU 数据。请把 MANUAL_WHU_ROOT 改为包含 train 和 val 的目录。"
        )

    valid.sort(
        key=lambda c: (
            "whu" not in str(c[0]).lower(),
            len(str(c[0]))
        )
    )

    return valid[0]


TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL = locate_whu()

print("TRAIN_IMG:", TRAIN_IMG)
print("TRAIN_LBL:", TRAIN_LBL)
print("VAL_IMG  :", VAL_IMG)
print("VAL_LBL  :", VAL_LBL)

# =========================
# 3. Dataset
# =========================
class WHUDataset(Dataset):
    def __init__(
        self,
        img_dir,
        lbl_dir,
        img_size=256
    ):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.img_size = img_size

        valid_ext = {
            ".png",
            ".jpg",
            ".jpeg",
            ".tif",
            ".tiff",
        }

        self.names = sorted(
            p.name
            for p in self.img_dir.iterdir()
            if p.is_file()
            and p.suffix.lower() in valid_ext
        )

        if not self.names:
            raise RuntimeError(
                f"没有找到图像：{self.img_dir}"
            )

        missing = [
            name
            for name in self.names
            if not (self.lbl_dir / name).exists()
        ]

        if missing:
            raise FileNotFoundError(
                f"缺少同名标签，示例：{missing[:5]}"
            )

    def __len__(self):
        return len(self.names)

    @staticmethod
    def make_whu_edge(mask_np):
        mask_u8 = mask_np.astype(np.uint8) * 255

        kernel = np.ones(
            (3, 3),
            np.uint8
        )

        dilated = cv2.dilate(
            mask_u8,
            kernel,
            iterations=1
        )

        return (
            (dilated - mask_u8) > 0
        ).astype(np.float32)

    def __getitem__(self, idx):
        name = self.names[idx]

        image = Image.open(
            self.img_dir / name
        ).convert("RGB")

        label = Image.open(
            self.lbl_dir / name
        ).convert("L")

        image = image.resize(
            (self.img_size, self.img_size),
            Image.BILINEAR
        )

        label = label.resize(
            (self.img_size, self.img_size),
            Image.NEAREST
        )

        image_tensor = T.ToTensor()(image)

        label_np = (
            np.array(label) > 127
        ).astype(np.uint8)

        edge_np = self.make_whu_edge(label_np)

        return (
            image_tensor,
            torch.from_numpy(label_np).long(),
            torch.from_numpy(edge_np).float(),
            name,
        )


train_ds = WHUDataset(
    TRAIN_IMG,
    TRAIN_LBL,
    IMG_SIZE
)

val_ds = WHUDataset(
    VAL_IMG,
    VAL_LBL,
    IMG_SIZE
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))

# =========================
# 4. 模型结构
# =========================
class EdgeBranch(nn.Module):
    def __init__(
        self,
        in_channels=3,
        out_channels=32
    ):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

        self.edge_head = nn.Conv2d(
            out_channels,
            1,
            kernel_size=1
        )

    def forward(self, x):
        feat = self.conv(x)
        edge_logits = self.edge_head(feat)
        return feat, edge_logits


class ResNetSemanticBranch(nn.Module):
    def __init__(
        self,
        out_channels=64
    ):
        super().__init__()

        weights = (
            models.ResNet18_Weights.DEFAULT
            if USE_IMAGENET_WEIGHTS
            else None
        )

        try:
            resnet = models.resnet18(
                weights=weights
            )
        except Exception as exc:
            raise RuntimeError(
                "ResNet18 权重加载失败。请打开 Kaggle Internet，"
                "或者将 USE_IMAGENET_WEIGHTS=False。"
            ) from exc

        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        self.smooth = nn.Sequential(
            nn.Conv2d(
                512,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.smooth(feat)

        return F.interpolate(
            feat,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )


class EdgeEnhancedNet(nn.Module):
    def __init__(
        self,
        use_edge=True
    ):
        super().__init__()

        self.use_edge = use_edge
        self.semantic_branch = ResNetSemanticBranch(64)

        if use_edge:
            self.edge_branch = EdgeBranch(
                in_channels=3,
                out_channels=32
            )
            fusion_in = 96
        else:
            self.edge_branch = None
            fusion_in = 64

        self.fusion = nn.Sequential(
            nn.Conv2d(
                fusion_in,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                64,
                2,
                kernel_size=1
            ),
        )

    def forward(self, x):
        semantic_feat = self.semantic_branch(x)

        edge_logits = None

        if self.use_edge:
            edge_feat, edge_logits = self.edge_branch(x)

            fused = torch.cat(
                [semantic_feat, edge_feat],
                dim=1
            )
        else:
            fused = semantic_feat

        seg_logits = self.fusion(fused)

        return seg_logits, edge_logits


# =========================
# 5. 损失函数
# =========================
ce_loss = nn.CrossEntropyLoss()
edge_bce = nn.BCEWithLogitsLoss()

def dice_loss_building(
    logits,
    target,
    eps=1e-6
):
    prob = torch.softmax(
        logits,
        dim=1
    )[:, 1]

    gt = (
        target == 1
    ).float()

    inter = (
        prob * gt
    ).sum(dim=(1, 2))

    denom = (
        prob.sum(dim=(1, 2))
        + gt.sum(dim=(1, 2))
    )

    dice = (
        2.0 * inter + eps
    ) / (
        denom + eps
    )

    return (
        1.0 - dice
    ).mean()


# =========================
# 6. 评价指标
# =========================
def get_boundary(mask):
    mask_u8 = mask.astype(np.uint8) * 255

    kernel = np.ones(
        (3, 3),
        np.uint8
    )

    dilated = cv2.dilate(
        mask_u8,
        kernel,
        iterations=1
    )

    eroded = cv2.erode(
        mask_u8,
        kernel,
        iterations=1
    )

    return (
        (dilated - eroded) > 0
    ).astype(np.uint8)


def boundary_metrics(
    gt,
    pred,
    tolerance=2
):
    gt_boundary = get_boundary(gt)
    pred_boundary = get_boundary(pred)

    intersection = np.logical_and(
        gt_boundary,
        pred_boundary
    ).sum()

    union = np.logical_or(
        gt_boundary,
        pred_boundary
    ).sum()

    boundary_iou = (
        intersection / (union + 1e-8)
    )

    kernel = np.ones(
        (
            2 * tolerance + 1,
            2 * tolerance + 1
        ),
        np.uint8
    )

    gt_dilated = cv2.dilate(
        gt_boundary,
        kernel
    )

    pred_dilated = cv2.dilate(
        pred_boundary,
        kernel
    )

    matched_pred = np.logical_and(
        pred_boundary == 1,
        gt_dilated == 1
    ).sum()

    matched_gt = np.logical_and(
        gt_boundary == 1,
        pred_dilated == 1
    ).sum()

    precision = matched_pred / (
        (pred_boundary == 1).sum()
        + 1e-8
    )

    recall = matched_gt / (
        (gt_boundary == 1).sum()
        + 1e-8
    )

    boundary_f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return float(boundary_f1), float(boundary_iou)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    boundary_f1_list = []
    boundary_iou_list = []

    for images, labels, _, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        seg_logits, _ = model(images)

        pred = torch.argmax(
            seg_logits,
            dim=1
        ).cpu().numpy().astype(np.uint8)

        gt = labels.numpy().astype(np.uint8)

        tp += np.logical_and(
            pred == 1,
            gt == 1
        ).sum()

        fp += np.logical_and(
            pred == 1,
            gt == 0
        ).sum()

        fn += np.logical_and(
            pred == 0,
            gt == 1
        ).sum()

        tn += np.logical_and(
            pred == 0,
            gt == 0
        ).sum()

        for p, g in zip(pred, gt):
            bf1, biou = boundary_metrics(
                g,
                p,
                tolerance=2
            )

            boundary_f1_list.append(bf1)
            boundary_iou_list.append(biou)

    building_iou = tp / (
        tp + fp + fn + 1e-8
    )

    background_iou = tn / (
        tn + fp + fn + 1e-8
    )

    precision = tp / (
        tp + fp + 1e-8
    )

    recall = tp / (
        tp + fn + 1e-8
    )

    f1 = 2 * precision * recall / (
        precision + recall + 1e-8
    )

    return {
        "BuildingIoU": float(building_iou),
        "True_mIoU": float(
            (building_iou + background_iou) / 2
        ),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "BoundaryF1": float(
            np.mean(boundary_f1_list)
        ),
        "BoundaryIoU": float(
            np.mean(boundary_iou_list)
        ),
    }


def count_params_m(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ) / 1e6


# =========================
# 7. 开始训练
# =========================
seed_everything(SEED)

model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

last_path = CKPT_DIR / f"{EXPERIMENT}_last.pth"
best_path = CKPT_DIR / f"{EXPERIMENT}_best.pth"

best_iou = -1.0
history = []
start_epoch = 1

if RESUME and last_path.exists():
    checkpoint = torch.load(
        last_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

    best_iou = float(
        checkpoint.get(
            "best_iou",
            -1.0
        )
    )

    history = checkpoint.get(
        "history",
        []
    )

    start_epoch = int(
        checkpoint["epoch"]
    ) + 1

    print(
        f"从第 {start_epoch - 1} 轮恢复，"
        f"当前最佳 IoU={best_iou:.6f}"
    )

for epoch in range(
    start_epoch,
    EPOCHS + 1
):
    model.train()

    total_loss = 0.0
    total_seg_loss = 0.0
    total_edge_loss = 0.0
    batches = 0

    t0 = time.time()

    for images, labels, edges, _ in train_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        edges = edges.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            enabled=True
        ):
            seg_logits, edge_logits = model(images)

            loss_seg = (
                ce_loss(seg_logits, labels)
                + dice_loss_building(
                    seg_logits,
                    labels
                )
            )

            # X1 中保留 EdgeBranch，
            # 但不计算边缘监督损失
            loss_edge = torch.zeros(
                (),
                device=DEVICE
            )

            loss = loss_seg

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(
            loss.item()
        )

        total_seg_loss += float(
            loss_seg.item()
        )

        total_edge_loss += float(
            loss_edge.item()
        )

        batches += 1

    avg_loss = total_loss / max(
        batches,
        1
    )

    val_metrics = evaluate(
        model,
        val_loader
    )

    history.append({
        "epoch": epoch,
        "train_loss": avg_loss,
        "train_seg_loss": (
            total_seg_loss / max(batches, 1)
        ),
        "train_edge_loss": (
            total_edge_loss / max(batches, 1)
        ),
        **val_metrics,
    })

    if val_metrics["BuildingIoU"] > best_iou:
        best_iou = val_metrics["BuildingIoU"]

        torch.save(
            model.state_dict(),
            best_path
        )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_iou": best_iou,
            "history": history,
            "experiment": EXPERIMENT,
        },
        last_path
    )

    pd.DataFrame(history).to_csv(
        RESULT_DIR / f"{EXPERIMENT}_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss={avg_loss:.4f} | "
        f"IoU={val_metrics['BuildingIoU'] * 100:.2f}% | "
        f"F1={val_metrics['F1'] * 100:.2f}% | "
        f"BF1={val_metrics['BoundaryF1'] * 100:.2f}%"
    )

# =========================
# 8. 最终评估
# =========================
if not best_path.exists():
    raise FileNotFoundError(
        f"没有找到最佳权重：{best_path}"
    )

best_model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

best_model.load_state_dict(
    torch.load(
        best_path,
        map_location=DEVICE
    )
)

metrics = evaluate(
    best_model,
    val_loader
)

metrics_pct = {
    k: v * 100.0
    for k, v in metrics.items()
}

config = {
    "experiment": EXPERIMENT,
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "lr": LR,
    "use_edge": USE_EDGE,
    "edge_lambda": EDGE_LAMBDA,
    "main_loss": "CrossEntropy + Dice",
    "edge_loss": "disabled",
    "whu_edge_label": "3x3 Dilate(M)-M",
    "best_checkpoint_rule": (
        "maximum validation Building IoU"
    ),
    "postprocessing": "none",
    "imagenet_pretrained": USE_IMAGENET_WEIGHTS,
    "params_M": count_params_m(best_model),
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "train_img": str(TRAIN_IMG),
    "val_img": str(VAL_IMG),
    "device": torch.cuda.get_device_name(0),
}

(RESULT_DIR / f"{EXPERIMENT}_config.json").write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

(RESULT_DIR / f"{EXPERIMENT}_metrics.json").write_text(
    json.dumps(
        metrics_pct,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

report_lines = [
    f"WHU formal ablation experiment: {EXPERIMENT}",
    "=" * 70,
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    "",
    "Best checkpoint validation metrics (%)",
]

report_lines.extend(
    f"{k}: {v:.4f}"
    for k, v in metrics_pct.items()
)

report = "\n".join(report_lines)

(RESULT_DIR / f"{EXPERIMENT}_report.txt").write_text(
    report,
    encoding="utf-8"
)

# =========================
# 9. 保存训练曲线
# =========================
hist = pd.DataFrame(history)

plt.figure(figsize=(7.2, 4.6))

plt.plot(
    hist["epoch"],
    hist["train_loss"],
    linewidth=1.8
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title(f"{EXPERIMENT} Training Curve")
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_training_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# =========================
# 10. 保存指标图
# =========================
metric_names = [
    "BuildingIoU",
    "F1",
    "BoundaryF1",
    "BoundaryIoU",
]

metric_values = [
    metrics_pct[k]
    for k in metric_names
]

plt.figure(figsize=(7.2, 4.6))

bars = plt.bar(
    metric_names,
    metric_values
)

plt.ylim(0, 100)
plt.ylabel("Score (%)")
plt.title(f"{EXPERIMENT} Validation Metrics")

for bar, value in zip(
    bars,
    metric_values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# =========================
# 11. 打包结果
# =========================
archive_path = Path(
    "/kaggle/working"
) / f"{EXPERIMENT}_artifacts.zip"

with zipfile.ZipFile(
    archive_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for file in OUT_ROOT.rglob("*"):
        if file.is_file():
            zf.write(
                file,
                file.relative_to(
                    OUT_ROOT.parent
                )
            )

print("\n" + report)
print("\n结果压缩包：", archive_path)

Device: cuda
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
Experiment: X1_Edge_NoSupervision_seed2026
USE_EDGE: True
EDGE_LAMBDA: 0.0
OUT_ROOT: /kaggle/working/WHU_X1_Edge_NoSupervision_seed2026_final
TRAIN_IMG: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
TRAIN_LBL: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/label
VAL_IMG  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
VAL_LBL  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/label
Train samples: 4736
Val samples  : 1036
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 186MB/s]


Epoch 01/50 | Loss=0.5317 | IoU=64.79% | F1=78.63% | BF1=49.56%
Epoch 02/50 | Loss=0.4329 | IoU=73.66% | F1=84.83% | BF1=55.15%
Epoch 03/50 | Loss=0.3991 | IoU=72.46% | F1=84.03% | BF1=50.88%
Epoch 04/50 | Loss=0.3772 | IoU=56.39% | F1=72.12% | BF1=38.26%
Epoch 05/50 | Loss=0.3600 | IoU=78.35% | F1=87.86% | BF1=54.82%
Epoch 06/50 | Loss=0.3368 | IoU=80.86% | F1=89.42% | BF1=57.51%
Epoch 07/50 | Loss=0.3379 | IoU=75.67% | F1=86.15% | BF1=53.89%
Epoch 08/50 | Loss=0.3186 | IoU=78.21% | F1=87.78% | BF1=56.19%
Epoch 09/50 | Loss=0.3560 | IoU=63.57% | F1=77.73% | BF1=43.40%
Epoch 10/50 | Loss=0.3612 | IoU=69.67% | F1=82.13% | BF1=51.33%
Epoch 11/50 | Loss=0.3111 | IoU=78.57% | F1=88.00% | BF1=53.98%
Epoch 12/50 | Loss=0.2895 | IoU=66.07% | F1=79.57% | BF1=47.47%
Epoch 13/50 | Loss=0.3011 | IoU=79.82% | F1=88.78% | BF1=55.59%
Epoch 14/50 | Loss=0.2914 | IoU=81.08% | F1=89.55% | BF1=55.65%
Epoch 15/50 | Loss=0.2536 | IoU=70.60% | F1=82.77% | BF1=52.98%
Epoch 16/50 | Loss=0.2722 | IoU=81.30% |

In [ ]:
import os, random, json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models

import matplotlib.pyplot as plt


# ============================================================
# 0. 固定实验参数
# ============================================================
SEED = 2026
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2

USE_IMAGENET_WEIGHTS = True
RESUME = False


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 GPU，请在 Kaggle Notebook Settings -> Accelerator 中选择 GPU。"
    )

DEVICE = torch.device("cuda")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 1. X2 实验配置
# ============================================================
EXPERIMENT = "X2_Edge_Supervision_seed2026"

# X2：保留 EdgeBranch，并启用边缘监督
USE_EDGE = True
EDGE_LAMBDA = 0.2

if not USE_EDGE:
    raise ValueError("X2 必须设置 USE_EDGE=True")

if EDGE_LAMBDA <= 0:
    raise ValueError("X2 必须设置 EDGE_LAMBDA>0")


OUT_ROOT = Path("/kaggle/working") / f"WHU_{EXPERIMENT}_final"
CKPT_DIR = OUT_ROOT / "checkpoints"
RESULT_DIR = OUT_ROOT / "results"
FIG_DIR = OUT_ROOT / "figures"

for p in (CKPT_DIR, RESULT_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT)
print("USE_EDGE:", USE_EDGE)
print("EDGE_LAMBDA:", EDGE_LAMBDA)
print("OUT_ROOT:", OUT_ROOT)


# ============================================================
# 2. 自动寻找 WHU 数据
# ============================================================
MANUAL_WHU_ROOT = ""


def locate_whu():
    if MANUAL_WHU_ROOT.strip():
        root = Path(MANUAL_WHU_ROOT)

        candidates = [(
            root / "train" / "image",
            root / "train" / "label",
            root / "val" / "image",
            root / "val" / "label",
        )]
    else:
        candidates = []

        for top in (
            Path("/kaggle/input"),
            Path("/kaggle/working"),
        ):
            if not top.exists():
                continue

            for train_dir in top.rglob("train"):
                candidates.append((
                    train_dir / "image",
                    train_dir / "label",
                    train_dir.parent / "val" / "image",
                    train_dir.parent / "val" / "label",
                ))

    valid = [
        c for c in candidates
        if all(p.is_dir() for p in c)
    ]

    if not valid:
        raise FileNotFoundError(
            "没有找到 WHU 数据。请把 MANUAL_WHU_ROOT 改为包含 train 和 val 的目录。"
        )

    valid.sort(
        key=lambda c: (
            "whu" not in str(c[0]).lower(),
            len(str(c[0]))
        )
    )

    return valid[0]


TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL = locate_whu()

print("TRAIN_IMG:", TRAIN_IMG)
print("TRAIN_LBL:", TRAIN_LBL)
print("VAL_IMG  :", VAL_IMG)
print("VAL_LBL  :", VAL_LBL)


# ============================================================
# 3. Dataset
# ============================================================
class WHUDataset(Dataset):
    def __init__(
        self,
        img_dir,
        lbl_dir,
        img_size=256
    ):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.img_size = img_size

        valid_ext = {
            ".png",
            ".jpg",
            ".jpeg",
            ".tif",
            ".tiff",
        }

        self.names = sorted(
            p.name
            for p in self.img_dir.iterdir()
            if p.is_file()
            and p.suffix.lower() in valid_ext
        )

        if not self.names:
            raise RuntimeError(
                f"没有找到图像：{self.img_dir}"
            )

        missing = [
            name
            for name in self.names
            if not (self.lbl_dir / name).exists()
        ]

        if missing:
            raise FileNotFoundError(
                f"缺少同名标签，示例：{missing[:5]}"
            )

    def __len__(self):
        return len(self.names)

    @staticmethod
    def make_whu_edge(mask_np):
        mask_u8 = mask_np.astype(np.uint8) * 255

        kernel = np.ones(
            (3, 3),
            np.uint8
        )

        dilated = cv2.dilate(
            mask_u8,
            kernel,
            iterations=1
        )

        return (
            (dilated - mask_u8) > 0
        ).astype(np.float32)

    def __getitem__(self, idx):
        name = self.names[idx]

        image = Image.open(
            self.img_dir / name
        ).convert("RGB")

        label = Image.open(
            self.lbl_dir / name
        ).convert("L")

        image = image.resize(
            (self.img_size, self.img_size),
            Image.BILINEAR
        )

        label = label.resize(
            (self.img_size, self.img_size),
            Image.NEAREST
        )

        image_tensor = T.ToTensor()(image)

        label_np = (
            np.array(label) > 127
        ).astype(np.uint8)

        edge_np = self.make_whu_edge(label_np)

        return (
            image_tensor,
            torch.from_numpy(label_np).long(),
            torch.from_numpy(edge_np).float(),
            name,
        )


train_ds = WHUDataset(
    TRAIN_IMG,
    TRAIN_LBL,
    IMG_SIZE
)

val_ds = WHUDataset(
    VAL_IMG,
    VAL_LBL,
    IMG_SIZE
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))


# ============================================================
# 4. 模型结构
# ============================================================
class EdgeBranch(nn.Module):
    def __init__(
        self,
        in_channels=3,
        out_channels=32
    ):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

        self.edge_head = nn.Conv2d(
            out_channels,
            1,
            kernel_size=1
        )

    def forward(self, x):
        feat = self.conv(x)
        edge_logits = self.edge_head(feat)
        return feat, edge_logits


class ResNetSemanticBranch(nn.Module):
    def __init__(
        self,
        out_channels=64
    ):
        super().__init__()

        weights = (
            models.ResNet18_Weights.DEFAULT
            if USE_IMAGENET_WEIGHTS
            else None
        )

        try:
            resnet = models.resnet18(
                weights=weights
            )
        except Exception as exc:
            raise RuntimeError(
                "ResNet18 权重加载失败。请打开 Kaggle Internet，"
                "或者将 USE_IMAGENET_WEIGHTS=False。"
            ) from exc

        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        self.smooth = nn.Sequential(
            nn.Conv2d(
                512,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.smooth(feat)

        return F.interpolate(
            feat,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )


class EdgeEnhancedNet(nn.Module):
    def __init__(
        self,
        use_edge=True
    ):
        super().__init__()

        self.use_edge = use_edge
        self.semantic_branch = ResNetSemanticBranch(64)

        if use_edge:
            self.edge_branch = EdgeBranch(
                in_channels=3,
                out_channels=32
            )
            fusion_in = 96
        else:
            self.edge_branch = None
            fusion_in = 64

        self.fusion = nn.Sequential(
            nn.Conv2d(
                fusion_in,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                64,
                2,
                kernel_size=1
            ),
        )

    def forward(self, x):
        semantic_feat = self.semantic_branch(x)

        edge_logits = None

        if self.use_edge:
            edge_feat, edge_logits = self.edge_branch(x)

            fused = torch.cat(
                [semantic_feat, edge_feat],
                dim=1
            )
        else:
            fused = semantic_feat

        seg_logits = self.fusion(fused)

        return seg_logits, edge_logits


# ============================================================
# 5. 损失函数
# ============================================================
ce_loss = nn.CrossEntropyLoss()
edge_bce = nn.BCEWithLogitsLoss()


def dice_loss_building(
    logits,
    target,
    eps=1e-6
):
    prob = torch.softmax(
        logits,
        dim=1
    )[:, 1]

    gt = (
        target == 1
    ).float()

    inter = (
        prob * gt
    ).sum(dim=(1, 2))

    denom = (
        prob.sum(dim=(1, 2))
        + gt.sum(dim=(1, 2))
    )

    dice = (
        2.0 * inter + eps
    ) / (
        denom + eps
    )

    return (
        1.0 - dice
    ).mean()


# ============================================================
# 6. 评价指标
# ============================================================
def get_boundary(mask):
    mask_u8 = mask.astype(np.uint8) * 255

    kernel = np.ones(
        (3, 3),
        np.uint8
    )

    dilated = cv2.dilate(
        mask_u8,
        kernel,
        iterations=1
    )

    eroded = cv2.erode(
        mask_u8,
        kernel,
        iterations=1
    )

    return (
        (dilated - eroded) > 0
    ).astype(np.uint8)


def boundary_metrics(
    gt,
    pred,
    tolerance=2
):
    gt_boundary = get_boundary(gt)
    pred_boundary = get_boundary(pred)

    intersection = np.logical_and(
        gt_boundary,
        pred_boundary
    ).sum()

    union = np.logical_or(
        gt_boundary,
        pred_boundary
    ).sum()

    boundary_iou = (
        intersection / (union + 1e-8)
    )

    kernel = np.ones(
        (
            2 * tolerance + 1,
            2 * tolerance + 1
        ),
        np.uint8
    )

    gt_dilated = cv2.dilate(
        gt_boundary,
        kernel
    )

    pred_dilated = cv2.dilate(
        pred_boundary,
        kernel
    )

    matched_pred = np.logical_and(
        pred_boundary == 1,
        gt_dilated == 1
    ).sum()

    matched_gt = np.logical_and(
        gt_boundary == 1,
        pred_dilated == 1
    ).sum()

    precision = matched_pred / (
        (pred_boundary == 1).sum()
        + 1e-8
    )

    recall = matched_gt / (
        (gt_boundary == 1).sum()
        + 1e-8
    )

    boundary_f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return float(boundary_f1), float(boundary_iou)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    boundary_f1_list = []
    boundary_iou_list = []

    for images, labels, _, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        seg_logits, _ = model(images)

        pred = torch.argmax(
            seg_logits,
            dim=1
        ).cpu().numpy().astype(np.uint8)

        gt = labels.numpy().astype(np.uint8)

        tp += np.logical_and(
            pred == 1,
            gt == 1
        ).sum()

        fp += np.logical_and(
            pred == 1,
            gt == 0
        ).sum()

        fn += np.logical_and(
            pred == 0,
            gt == 1
        ).sum()

        tn += np.logical_and(
            pred == 0,
            gt == 0
        ).sum()

        for p, g in zip(pred, gt):
            bf1, biou = boundary_metrics(
                g,
                p,
                tolerance=2
            )

            boundary_f1_list.append(bf1)
            boundary_iou_list.append(biou)

    building_iou = tp / (
        tp + fp + fn + 1e-8
    )

    background_iou = tn / (
        tn + fp + fn + 1e-8
    )

    precision = tp / (
        tp + fp + 1e-8
    )

    recall = tp / (
        tp + fn + 1e-8
    )

    f1 = 2 * precision * recall / (
        precision + recall + 1e-8
    )

    return {
        "BuildingIoU": float(building_iou),
        "True_mIoU": float(
            (building_iou + background_iou) / 2
        ),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "BoundaryF1": float(
            np.mean(boundary_f1_list)
        ),
        "BoundaryIoU": float(
            np.mean(boundary_iou_list)
        ),
    }


def count_params_m(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ) / 1e6


# ============================================================
# 7. 开始训练
# ============================================================
seed_everything(SEED)

model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

last_path = CKPT_DIR / f"{EXPERIMENT}_last.pth"
best_path = CKPT_DIR / f"{EXPERIMENT}_best.pth"

best_iou = -1.0
history = []
start_epoch = 1

if RESUME and last_path.exists():
    checkpoint = torch.load(
        last_path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

    best_iou = float(
        checkpoint.get(
            "best_iou",
            -1.0
        )
    )

    history = checkpoint.get(
        "history",
        []
    )

    start_epoch = int(
        checkpoint["epoch"]
    ) + 1

    print(
        f"从第 {start_epoch - 1} 轮恢复，"
        f"当前最佳 IoU={best_iou:.6f}"
    )


for epoch in range(
    start_epoch,
    EPOCHS + 1
):
    model.train()

    total_loss = 0.0
    total_seg_loss = 0.0
    total_edge_loss = 0.0
    batches = 0

    t0 = time.time()

    for images, labels, edges, _ in train_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        edges = edges.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            enabled=True
        ):
            seg_logits, edge_logits = model(images)

            loss_seg = (
                ce_loss(seg_logits, labels)
                + dice_loss_building(
                    seg_logits,
                    labels
                )
            )

            # X2 的核心：边缘监督
            loss_edge = edge_bce(
                edge_logits.squeeze(1),
                edges
            )

            loss = (
                loss_seg
                + EDGE_LAMBDA * loss_edge
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(
            loss.item()
        )

        total_seg_loss += float(
            loss_seg.item()
        )

        total_edge_loss += float(
            loss_edge.item()
        )

        batches += 1

    avg_loss = total_loss / max(
        batches,
        1
    )

    val_metrics = evaluate(
        model,
        val_loader
    )

    history.append({
        "epoch": epoch,
        "train_loss": avg_loss,
        "train_seg_loss": (
            total_seg_loss / max(batches, 1)
        ),
        "train_edge_loss": (
            total_edge_loss / max(batches, 1)
        ),
        **val_metrics,
    })

    if val_metrics["BuildingIoU"] > best_iou:
        best_iou = val_metrics["BuildingIoU"]

        torch.save(
            model.state_dict(),
            best_path
        )

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_iou": best_iou,
            "history": history,
            "experiment": EXPERIMENT,
        },
        last_path
    )

    pd.DataFrame(history).to_csv(
        RESULT_DIR / f"{EXPERIMENT}_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss={avg_loss:.4f} | "
        f"IoU={val_metrics['BuildingIoU'] * 100:.2f}% | "
        f"F1={val_metrics['F1'] * 100:.2f}% | "
        f"BF1={val_metrics['BoundaryF1'] * 100:.2f}%"
    )


# ============================================================
# 8. 最终评估
# ============================================================
if not best_path.exists():
    raise FileNotFoundError(
        f"没有找到最佳权重：{best_path}"
    )

best_model = EdgeEnhancedNet(
    use_edge=USE_EDGE
).to(DEVICE)

best_model.load_state_dict(
    torch.load(
        best_path,
        map_location=DEVICE
    )
)

metrics = evaluate(
    best_model,
    val_loader
)

metrics_pct = {
    k: v * 100.0
    for k, v in metrics.items()
}


config = {
    "experiment": EXPERIMENT,
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "lr": LR,
    "use_edge": USE_EDGE,
    "edge_lambda": EDGE_LAMBDA,
    "main_loss": "CrossEntropy + Dice",
    "edge_loss": "BCEWithLogitsLoss",
    "whu_edge_label": "3x3 Dilate(M)-M",
    "best_checkpoint_rule": (
        "maximum validation Building IoU"
    ),
    "postprocessing": "none",
    "imagenet_pretrained": USE_IMAGENET_WEIGHTS,
    "params_M": count_params_m(best_model),
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "train_img": str(TRAIN_IMG),
    "val_img": str(VAL_IMG),
    "device": torch.cuda.get_device_name(0),
}

(RESULT_DIR / f"{EXPERIMENT}_config.json").write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

(RESULT_DIR / f"{EXPERIMENT}_metrics.json").write_text(
    json.dumps(
        metrics_pct,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


report_lines = [
    f"WHU formal ablation experiment: {EXPERIMENT}",
    "=" * 70,
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    "",
    "Best checkpoint validation metrics (%)",
]

report_lines.extend(
    f"{k}: {v:.4f}"
    for k, v in metrics_pct.items()
)

report = "\n".join(report_lines)

(RESULT_DIR / f"{EXPERIMENT}_report.txt").write_text(
    report,
    encoding="utf-8"
)


# ============================================================
# 9. 保存训练曲线
# ============================================================
hist = pd.DataFrame(history)

plt.figure(figsize=(7.2, 4.6))

plt.plot(
    hist["epoch"],
    hist["train_loss"],
    linewidth=1.8
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title(f"{EXPERIMENT} Training Curve")
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_training_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 10. 保存指标图
# ============================================================
metric_names = [
    "BuildingIoU",
    "F1",
    "BoundaryF1",
    "BoundaryIoU",
]

metric_values = [
    metrics_pct[k]
    for k in metric_names
]

plt.figure(figsize=(7.2, 4.6))

bars = plt.bar(
    metric_names,
    metric_values
)

plt.ylim(0, 100)
plt.ylabel("Score (%)")
plt.title(f"{EXPERIMENT} Validation Metrics")

for bar, value in zip(
    bars,
    metric_values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    FIG_DIR / f"{EXPERIMENT}_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 11. 打包结果
# ============================================================
archive_path = Path(
    "/kaggle/working"
) / f"{EXPERIMENT}_artifacts.zip"

with zipfile.ZipFile(
    archive_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for file in OUT_ROOT.rglob("*"):
        if file.is_file():
            zf.write(
                file,
                file.relative_to(
                    OUT_ROOT.parent
                )
            )

print("\n" + report)
print("\n结果压缩包：", archive_path)

Device: cuda
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
Experiment: X2_Edge_Supervision_seed2026
USE_EDGE: True
EDGE_LAMBDA: 0.2
OUT_ROOT: /kaggle/working/WHU_X2_Edge_Supervision_seed2026_final
TRAIN_IMG: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
TRAIN_LBL: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/label
VAL_IMG  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
VAL_LBL  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/label
Train samples: 4736
Val samples  : 1036
Epoch 01/50 | Loss=0.5766 | IoU=66.11% | F1=79.60% | BF1=49.19%
Epoch 02/50 | Loss=0.4600 | IoU=70.17% | F1=82.47% | BF1=52.04%
Epoch 03/50 | Loss=0.4405 | IoU=53.65% | F1=69.83% | BF1=42.51%
Epoch 04/50 | Loss=0.4216 | IoU=71.87% | F1=83.64% | BF1=50.49%
Epoch 05/50 | Loss=0.3870 | IoU=71.71% | F1=83.53% | BF1=53.06%
Epoch 06/50 | Loss=0.3827 | IoU=81.61% | F1=89.87% | BF1=58.32%
Epoch 07/50 | Loss=0.3535 | IoU=80.91% | F1=89.45% | BF1=57.82%
Epoch 

In [1]:
# -*- coding: utf-8 -*-
"""
WHU formal comparison: U-Net with ImageNet-pretrained ResNet18 encoder

Kaggle usage:
1. Add the WHU dataset to the notebook.
2. Select a GPU accelerator.
3. Paste this whole file into one Code cell and run it.

The script automatically searches /kaggle/input for:
    train/image, train/label
    val/image,   val/label
    test/image,  test/label

If automatic search cannot find the data, set MANUAL_WHU_ROOT to the
dataset directory that directly contains train, val and test.
"""

import os

# These variables must be set before importing cv2, torch or matplotlib.
# They prevent the common Windows/local-runtime OpenMP conflict and are
# harmless on Kaggle Linux.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import json
import random
import time
import zipfile
from contextlib import nullcontext
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision import transforms as T


# Pillow 9+ exposes Image.Resampling; this fallback keeps older Kaggle
# images compatible as well.
RESAMPLING = getattr(Image, "Resampling", Image)


# Keep all libraries quiet and deterministic enough for a formal comparison.
try:
    cv2.setNumThreads(0)
except Exception:
    pass

try:
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass


# ============================================================
# 0. Formal experiment configuration
# ============================================================
EXPERIMENT = "UNet_ResNet18_Formal_Comparison"

SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2

# Set False to start from epoch 1 even if a previous last checkpoint exists.
AUTO_RESUME = True

# Keep True for the paper protocol. If Kaggle Internet is disabled, either
# attach resnet18-f37072fd.pth as a Kaggle dataset or enable Internet.
USE_IMAGENET_PRETRAINED = True
LOCAL_RESNET18_WEIGHTS = ""

# Usually leave this empty. Example:
# MANUAL_WHU_ROOT = "/kaggle/input/whu-data/whu_data"
MANUAL_WHU_ROOT = ""

OUT_ROOT = Path("/kaggle/working") / f"WHU_{EXPERIMENT}_final"
CKPT_DIR = OUT_ROOT / "checkpoints"
RESULT_DIR = OUT_ROOT / "results"
FIG_DIR = OUT_ROOT / "figures"

for directory in (CKPT_DIR, RESULT_DIR, FIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

BEST_PATH = CKPT_DIR / f"{EXPERIMENT}_best.pth"
LAST_PATH = CKPT_DIR / f"{EXPERIMENT}_last.pth"
ARCHIVE_PATH = Path("/kaggle/working") / f"{EXPERIMENT}_artifacts.zip"


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError(
        "当前没有启用 GPU。请在 Kaggle Notebook 的 Settings -> Accelerator "
        "中选择 GPU 后重新运行。"
    )

print("=" * 78)
print("WHU formal comparison:", EXPERIMENT)
print("=" * 78)
print("Device :", DEVICE)
print("PyTorch:", torch.__version__)
print("GPU    :", torch.cuda.get_device_name(0))


# ============================================================
# 1. WHU data discovery
# ============================================================
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


def find_named_dir(parent, names):
    """Return the first existing directory among aliases."""
    for name in names:
        candidate = parent / name
        if candidate.is_dir():
            return candidate
    return None


def build_split_paths(root):
    root = Path(root)
    result = []

    for split in ("train", "val", "test"):
        split_dir = root / split
        image_dir = find_named_dir(split_dir, ("image", "images"))
        label_dir = find_named_dir(split_dir, ("label", "labels", "mask", "masks"))
        if image_dir is None or label_dir is None:
            return None
        result.extend((image_dir, label_dir))

    return tuple(result)


def locate_whu():
    if MANUAL_WHU_ROOT.strip():
        paths = build_split_paths(MANUAL_WHU_ROOT.strip())
        if paths is None:
            raise FileNotFoundError(
                "MANUAL_WHU_ROOT 下未找到完整的 train/val/test "
                "image/label 目录："
                f"{MANUAL_WHU_ROOT}"
            )
        return paths

    search_roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    candidates = []
    seen = set()

    for search_root in search_roots:
        if not search_root.exists():
            continue

        for train_dir in search_root.rglob("train"):
            if not train_dir.is_dir():
                continue

            paths = build_split_paths(train_dir.parent)
            if paths is None:
                continue

            key = tuple(str(path.resolve()) for path in paths)
            if key not in seen:
                candidates.append(paths)
                seen.add(key)

    if not candidates:
        raise FileNotFoundError(
            "没有自动找到 WHU 数据集。\n"
            "请确认 Kaggle 已添加数据集，并检查其目录是否包含：\n"
            "train/image, train/label, val/image, val/label, "
            "test/image, test/label\n"
            "也可以把 MANUAL_WHU_ROOT 改成对应的根目录。"
        )

    # Prefer paths containing whu in their name, then prefer the shorter path.
    candidates.sort(
        key=lambda paths: (
            "whu" not in str(paths[0]).lower(),
            len(str(paths[0])),
        )
    )
    return candidates[0]


(
    TRAIN_IMG,
    TRAIN_LBL,
    VAL_IMG,
    VAL_LBL,
    TEST_IMG,
    TEST_LBL,
) = locate_whu()

print("\nWHU data:")
print("TRAIN_IMG:", TRAIN_IMG)
print("TRAIN_LBL:", TRAIN_LBL)
print("VAL_IMG  :", VAL_IMG)
print("VAL_LBL  :", VAL_LBL)
print("TEST_IMG :", TEST_IMG)
print("TEST_LBL :", TEST_LBL)


# ============================================================
# 2. Dataset
# ============================================================
class WHUDataset(Dataset):
    def __init__(self, image_dir, label_dir, img_size):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.img_size = int(img_size)

        self.names = sorted(
            path.name
            for path in self.image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS
        )
        if not self.names:
            raise RuntimeError(f"未找到图像文件：{self.image_dir}")

        missing = [
            name for name in self.names
            if not (self.label_dir / name).exists()
        ]
        if missing:
            preview = ", ".join(missing[:5])
            raise FileNotFoundError(
                f"{self.label_dir} 缺少 {len(missing)} 个同名标签。"
                f"示例：{preview}"
            )

        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image_path = self.image_dir / name
        label_path = self.label_dir / name

        image = Image.open(image_path).convert("RGB")
        label = Image.open(label_path).convert("L")

        image = image.resize(
            (self.img_size, self.img_size),
            RESAMPLING.BILINEAR,
        )
        label = label.resize(
            (self.img_size, self.img_size),
            RESAMPLING.NEAREST,
        )

        image_tensor = self.to_tensor(image)
        label_array = (np.asarray(label) > 127).astype(np.uint8)
        label_tensor = torch.from_numpy(label_array).long()

        return image_tensor, label_tensor, name


train_ds = WHUDataset(TRAIN_IMG, TRAIN_LBL, IMG_SIZE)
val_ds = WHUDataset(VAL_IMG, VAL_LBL, IMG_SIZE)
test_ds = WHUDataset(TEST_IMG, TEST_LBL, IMG_SIZE)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

PIN_MEMORY = DEVICE.type == "cuda"
COMMON_LOADER_ARGS = {
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
}
if NUM_WORKERS > 0:
    COMMON_LOADER_ARGS["persistent_workers"] = True

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    **COMMON_LOADER_ARGS,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **COMMON_LOADER_ARGS,
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **COMMON_LOADER_ARGS,
)

print("\nSamples:")
print("Train:", len(train_ds))
print("Val  :", len(val_ds))
print("Test :", len(test_ds))

EXPECTED_COUNTS = {"train": 4736, "val": 1036, "test": 2416}
actual_counts = {
    "train": len(train_ds),
    "val": len(val_ds),
    "test": len(test_ds),
}
if actual_counts != EXPECTED_COUNTS:
    print("\nWARNING: 样本数量与正式论文记录不一致。")
    print("Expected:", EXPECTED_COUNTS)
    print("Actual  :", actual_counts)
    print("请确认选中的确实是 WHU 数据目录。")
else:
    print("Dataset count check: OK")


# ============================================================
# 3. U-Net with ResNet18 encoder
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(
            in_channels + skip_channels,
            out_channels,
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        return self.conv(torch.cat([x, skip], dim=1))


def find_local_resnet18_weights():
    if LOCAL_RESNET18_WEIGHTS.strip():
        path = Path(LOCAL_RESNET18_WEIGHTS.strip())
        if not path.is_file():
            raise FileNotFoundError(
                f"LOCAL_RESNET18_WEIGHTS 不存在：{path}"
            )
        return path

    expected_name = "resnet18-f37072fd.pth"
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not root.exists():
            continue
        matches = list(root.rglob(expected_name))
        if matches:
            return matches[0]
    return None


def make_resnet18(pretrained):
    if not pretrained:
        return models.resnet18(weights=None)

    local_weights = find_local_resnet18_weights()
    if local_weights is not None:
        print("Using local ResNet18 weights:", local_weights)
        backbone = models.resnet18(weights=None)
        state = torch.load(local_weights, map_location="cpu")
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        backbone.load_state_dict(state, strict=True)
        return backbone

    try:
        print("Loading ImageNet-pretrained ResNet18 weights...")
        weights_enum = getattr(models, "ResNet18_Weights", None)
        if weights_enum is not None:
            return models.resnet18(weights=weights_enum.DEFAULT)
        # Compatibility with older torchvision versions.
        return models.resnet18(pretrained=True)
    except TypeError:
        # Compatibility with older torchvision versions.
        return models.resnet18(pretrained=True)
    except Exception as error:
        raise RuntimeError(
            "无法加载 ImageNet 预训练 ResNet18 权重。\n"
            "请在 Kaggle 右侧 Settings 中打开 Internet，或将 "
            "resnet18-f37072fd.pth 作为 Kaggle 数据集加入后重试。\n"
            "为了保持论文正式对比协议，脚本不会静默退回随机初始化。"
        ) from error


class UNetResNet18(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        backbone = make_resnet18(pretrained)

        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 128)
        self.up2 = UpBlock(128, 64, 64)
        self.up1 = UpBlock(64, 64, 64)
        self.final_refine = DoubleConv(64, 32)
        self.classifier = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        input_hw = x.shape[-2:]

        stem = self.relu(self.bn1(self.conv1(x)))  # H/2
        x = self.maxpool(stem)
        e1 = self.layer1(x)  # H/4
        e2 = self.layer2(e1)  # H/8
        e3 = self.layer3(e2)  # H/16
        e4 = self.layer4(e3)  # H/32

        d4 = self.up4(e4, e3)
        d3 = self.up3(d4, e2)
        d2 = self.up2(d3, e1)
        d1 = self.up1(d2, stem)
        d0 = F.interpolate(
            d1,
            size=input_hw,
            mode="bilinear",
            align_corners=False,
        )
        return self.classifier(self.final_refine(d0))


# ============================================================
# 4. Loss and metrics
# ============================================================
ce_loss = nn.CrossEntropyLoss()


def dice_loss_building(logits, target, eps=1e-6):
    probability = torch.softmax(logits, dim=1)[:, 1]
    ground_truth = (target == 1).float()
    intersection = (probability * ground_truth).sum(dim=(1, 2))
    denominator = (
        probability.sum(dim=(1, 2))
        + ground_truth.sum(dim=(1, 2))
    )
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice.mean()


def get_boundary(mask, kernel_size=3):
    mask = (mask.astype(np.uint8) * 255)
    kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
    dilated = cv2.dilate(mask, kernel, iterations=1)
    eroded = cv2.erode(mask, kernel, iterations=1)
    return ((dilated - eroded) > 0).astype(np.uint8)


def boundary_metrics(gt, pred, tolerance=2):
    gt_boundary = get_boundary(gt, 3)
    pred_boundary = get_boundary(pred, 3)

    intersection = np.logical_and(
        gt_boundary,
        pred_boundary,
    ).sum()
    union = np.logical_or(gt_boundary, pred_boundary).sum()
    boundary_iou = intersection / (union + 1e-8)

    kernel = np.ones(
        (2 * tolerance + 1, 2 * tolerance + 1),
        dtype=np.uint8,
    )
    gt_dilated = cv2.dilate(gt_boundary, kernel, iterations=1)
    pred_dilated = cv2.dilate(pred_boundary, kernel, iterations=1)

    matched_pred = np.logical_and(
        pred_boundary == 1,
        gt_dilated == 1,
    ).sum()
    matched_gt = np.logical_and(
        gt_boundary == 1,
        pred_dilated == 1,
    ).sum()

    boundary_precision = matched_pred / ((pred_boundary == 1).sum() + 1e-8)
    boundary_recall = matched_gt / ((gt_boundary == 1).sum() + 1e-8)
    boundary_f1 = (
        2.0 * boundary_precision * boundary_recall
        / (boundary_precision + boundary_recall + 1e-8)
    )
    return float(boundary_f1), float(boundary_iou)


@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    tp = fp = fn = tn = 0
    boundary_f1_values = []
    boundary_iou_values = []

    for images, labels, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        prediction = torch.argmax(model(images), dim=1)

        pred_np = prediction.cpu().numpy().astype(np.uint8)
        gt_np = labels.cpu().numpy().astype(np.uint8)

        tp += np.logical_and(pred_np == 1, gt_np == 1).sum()
        fp += np.logical_and(pred_np == 1, gt_np == 0).sum()
        fn += np.logical_and(pred_np == 0, gt_np == 1).sum()
        tn += np.logical_and(pred_np == 0, gt_np == 0).sum()

        for pred_item, gt_item in zip(pred_np, gt_np):
            bf1, biou = boundary_metrics(gt_item, pred_item, 2)
            boundary_f1_values.append(bf1)
            boundary_iou_values.append(biou)

    building_iou = tp / (tp + fp + fn + 1e-8)
    background_iou = tn / (tn + fp + fn + 1e-8)
    true_miou = (building_iou + background_iou) / 2.0
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2.0 * precision * recall / (precision + recall + 1e-8)

    return {
        "BuildingIoU": float(building_iou),
        "True_mIoU": float(true_miou),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "BoundaryF1": float(np.mean(boundary_f1_values)),
        "BoundaryIoU": float(np.mean(boundary_iou_values)),
    }


def count_params_m(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    ) / 1e6


# ============================================================
# 5. AMP compatibility
# ============================================================
AMP_ENABLED = DEVICE.type == "cuda"

if AMP_ENABLED:
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=True)

        def autocast_context():
            return torch.amp.autocast(device_type="cuda", enabled=True)

    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=True)

        def autocast_context():
            return torch.cuda.amp.autocast(enabled=True)
else:
    scaler = torch.amp.GradScaler("cuda", enabled=False)

    def autocast_context():
        return nullcontext()


# ============================================================
# 6. Build model and optimizer
# ============================================================
seed_everything(SEED)
model = UNetResNet18(
    num_classes=2,
    pretrained=USE_IMAGENET_PRETRAINED,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
params_m = count_params_m(model)

print("\nModel:")
print("U-Net (ImageNet-pretrained ResNet18 encoder)")
print("Params_M:", f"{params_m:.6f}")
print("AMP     :", AMP_ENABLED)


# ============================================================
# 7. Optional resume from last checkpoint
# ============================================================
start_epoch = 1
best_iou = -1.0
history = []

if AUTO_RESUME and LAST_PATH.exists():
    print("\n发现 last checkpoint，尝试继续训练：")
    print(LAST_PATH)

    last = torch.load(LAST_PATH, map_location=DEVICE)
    model.load_state_dict(last["model_state_dict"], strict=True)
    optimizer.load_state_dict(last["optimizer_state_dict"])

    saved_scaler = last.get("scaler_state_dict")
    if saved_scaler:
        scaler.load_state_dict(saved_scaler)

    best_iou = float(last.get("best_iou", -1.0))
    history = list(last.get("history", []))
    start_epoch = int(last.get("epoch", 0)) + 1

    print(f"Resume from epoch {start_epoch}")
    print(f"Current best val Building IoU: {best_iou * 100:.4f}%")


# ============================================================
# 8. Formal training
# ============================================================
for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    batch_count = 0
    start_time = time.time()

    for images, labels, _ in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast_context():
            logits = model(images)
            loss = ce_loss(logits, labels) + dice_loss_building(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(loss.item())
        batch_count += 1

    average_loss = total_loss / max(batch_count, 1)
    val_metrics = evaluate(model, val_loader)
    history.append(
        {
            "epoch": epoch,
            "train_loss": average_loss,
            **val_metrics,
        }
    )

    if val_metrics["BuildingIoU"] > best_iou:
        best_iou = val_metrics["BuildingIoU"]
        torch.save(model.state_dict(), BEST_PATH)

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_iou": best_iou,
            "history": history,
            "experiment": EXPERIMENT,
        },
        LAST_PATH,
    )

    pd.DataFrame(history).to_csv(
        RESULT_DIR / f"{EXPERIMENT}_history.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss={average_loss:.4f} | "
        f"ValIoU={val_metrics['BuildingIoU'] * 100:.2f}% | "
        f"ValF1={val_metrics['F1'] * 100:.2f}% | "
        f"ValBF1={val_metrics['BoundaryF1'] * 100:.2f}% | "
        f"BestIoU={best_iou * 100:.2f}% | "
        f"{time.time() - start_time:.1f}s"
    )


# ============================================================
# 9. Load best model and evaluate validation/test once
# ============================================================
if not BEST_PATH.exists():
    raise FileNotFoundError(f"没有找到最佳权重：{BEST_PATH}")

# No need to load ImageNet weights a second time; the checkpoint contains
# the complete trained model.
best_model = UNetResNet18(
    num_classes=2,
    pretrained=False,
).to(DEVICE)
best_model.load_state_dict(
    torch.load(BEST_PATH, map_location=DEVICE),
    strict=True,
)
best_model.eval()

val_metrics = evaluate(best_model, val_loader)
test_metrics = evaluate(best_model, test_loader)
val_metrics_pct = {key: value * 100.0 for key, value in val_metrics.items()}
test_metrics_pct = {key: value * 100.0 for key, value in test_metrics.items()}


# ============================================================
# 10. Save formal results
# ============================================================
config = {
    "experiment": EXPERIMENT,
    "paper_method_name": "U-Net (ResNet18 encoder)",
    "comparison_role": "formal WHU comparison baseline",
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "lr": LR,
    "main_loss": "CrossEntropy + Dice",
    "edge_loss": "disabled",
    "best_checkpoint_rule": "maximum validation Building IoU",
    "prediction_rule": "argmax over 2-class logits",
    "postprocessing": "none",
    "encoder": "ImageNet-pretrained ResNet18",
    "input_normalization": (
        "ToTensor only; no additional ImageNet normalization, "
        "matching X0/X1/X2 formal protocol"
    ),
    "boundary_definition": "3x3 morphological boundary",
    "boundary_f1_tolerance_px": 2,
    "boundary_iou": "strict intersection-over-union",
    "params_M": count_params_m(best_model),
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "test_samples": len(test_ds),
    "train_img": str(TRAIN_IMG),
    "val_img": str(VAL_IMG),
    "test_img": str(TEST_IMG),
    "device": torch.cuda.get_device_name(0),
}

(RESULT_DIR / f"{EXPERIMENT}_config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(RESULT_DIR / f"{EXPERIMENT}_val_metrics.json").write_text(
    json.dumps(val_metrics_pct, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(RESULT_DIR / f"{EXPERIMENT}_test_metrics.json").write_text(
    json.dumps(test_metrics_pct, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

report_lines = [
    f"WHU formal comparison experiment: {EXPERIMENT}",
    "=" * 78,
    "",
    "Experiment config",
    json.dumps(config, ensure_ascii=False, indent=2),
    "",
    "Best checkpoint VALIDATION metrics (%)",
]
report_lines.extend(
    f"{key}: {value:.4f}" for key, value in val_metrics_pct.items()
)
report_lines.extend(["", "Independent TEST metrics (%)"])
report_lines.extend(
    f"{key}: {value:.4f}" for key, value in test_metrics_pct.items()
)
report = "\n".join(report_lines)
report_path = RESULT_DIR / f"{EXPERIMENT}_report.txt"
report_path.write_text(report, encoding="utf-8")

summary_df = pd.DataFrame(
    [
        {
            "Method": "U-Net (ResNet18 encoder)",
            "Split": "Validation",
            **val_metrics_pct,
            "Params_M": count_params_m(best_model),
        },
        {
            "Method": "U-Net (ResNet18 encoder)",
            "Split": "Independent Test",
            **test_metrics_pct,
            "Params_M": count_params_m(best_model),
        },
    ]
)
summary_path = RESULT_DIR / f"{EXPERIMENT}_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")


# ============================================================
# 11. Training curves
# ============================================================
history_df = pd.DataFrame(history)
if not history_df.empty:
    plt.figure(figsize=(7.2, 4.6))
    plt.plot(
        history_df["epoch"],
        history_df["train_loss"],
        linewidth=1.8,
    )
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title("U-Net (ResNet18 encoder) Training Loss")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(
        FIG_DIR / f"{EXPERIMENT}_training_loss.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()

    plt.figure(figsize=(7.2, 4.6))
    plt.plot(
        history_df["epoch"],
        history_df["BuildingIoU"] * 100,
        linewidth=1.8,
        label="Building IoU",
    )
    plt.plot(
        history_df["epoch"],
        history_df["F1"] * 100,
        linewidth=1.8,
        label="F1",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Validation score (%)")
    plt.title("U-Net Validation Performance")
    plt.grid(alpha=0.2)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        FIG_DIR / f"{EXPERIMENT}_validation_curve.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()


# ============================================================
# 12. Package all outputs as a Kaggle downloadable artifact
# ============================================================
with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for file_path in OUT_ROOT.rglob("*"):
        if file_path.is_file():
            archive.write(
                file_path,
                file_path.relative_to(OUT_ROOT.parent),
            )

print("\n" + report)
print("\n" + "=" * 78)
print("正式 U-Net 对比实验完成")
print("=" * 78)
print("Best checkpoint :", BEST_PATH)
print("Report          :", report_path)
print("Summary CSV     :", summary_path)
print("Artifacts ZIP   :", ARCHIVE_PATH)
print("\n论文表格建议使用 Independent Test 行：")
for key, value in test_metrics_pct.items():
    print(f"{key}: {value:.4f}")


WHU formal comparison: UNet_ResNet18_Formal_Comparison
Device : cuda
PyTorch: 2.10.0+cu128
GPU    : Tesla T4

WHU data:
TRAIN_IMG: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
TRAIN_LBL: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/label
VAL_IMG  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
VAL_LBL  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/label
TEST_IMG : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/test/image
TEST_LBL : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/test/label

Samples:
Train: 4736
Val  : 1036
Test : 2416
Dataset count check: OK
Loading ImageNet-pretrained ResNet18 weights...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 220MB/s]



Model:
U-Net (ImageNet-pretrained ResNet18 encoder)
Params_M: 14.413570
AMP     : True
Epoch 01/50 | Loss=0.4783 | ValIoU=73.37% | ValF1=84.64% | ValBF1=56.34% | BestIoU=73.37% | 63.0s
Epoch 02/50 | Loss=0.3468 | ValIoU=79.98% | ValF1=88.88% | ValBF1=58.88% | BestIoU=79.98% | 48.6s
Epoch 03/50 | Loss=0.3267 | ValIoU=83.00% | ValF1=90.71% | ValBF1=60.29% | BestIoU=83.00% | 49.3s
Epoch 04/50 | Loss=0.3032 | ValIoU=79.78% | ValF1=88.75% | ValBF1=59.66% | BestIoU=83.00% | 49.0s
Epoch 05/50 | Loss=0.2919 | ValIoU=82.52% | ValF1=90.43% | ValBF1=60.33% | BestIoU=83.00% | 49.2s
Epoch 06/50 | Loss=0.2854 | ValIoU=83.05% | ValF1=90.74% | ValBF1=61.41% | BestIoU=83.05% | 49.1s
Epoch 07/50 | Loss=0.2726 | ValIoU=85.30% | ValF1=92.07% | ValBF1=62.50% | BestIoU=85.30% | 49.2s
Epoch 08/50 | Loss=0.2725 | ValIoU=84.49% | ValF1=91.59% | ValBF1=62.20% | BestIoU=85.30% | 49.0s
Epoch 09/50 | Loss=0.2582 | ValIoU=85.65% | ValF1=92.27% | ValBF1=61.46% | BestIoU=85.65% | 49.1s
Epoch 10/50 | Loss=0.2592 | Va

In [1]:
# ============================================================
# WHU Formal Comparison:
# DeepLabV3+ (ResNet18 encoder)
# ============================================================

import os

# Kaggle-safe defaults. Harmless elsewhere, helpful on Linux and local Windows runtimes.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import random
import json
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
RESAMPLING = getattr(Image, "Resampling", Image)
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# 0. 固定正式实验参数
# ============================================================
EXPERIMENT = "DeepLabV3Plus_ResNet18_Formal_Comparison"

SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2

AUTO_RESUME = True

# 自动搜索失败时再手动填写：
# MANUAL_WHU_ROOT = "/kaggle/input/datasets/xiaomaoperson/whu-data/whu_data"
MANUAL_WHU_ROOT = ""

OUT_ROOT = Path("/kaggle/working") / f"WHU_{EXPERIMENT}_final"
CKPT_DIR = OUT_ROOT / "checkpoints"
RESULT_DIR = OUT_ROOT / "results"
FIG_DIR = OUT_ROOT / "figures"

for p in (CKPT_DIR, RESULT_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

BEST_PATH = CKPT_DIR / f"{EXPERIMENT}_best.pth"
LAST_PATH = CKPT_DIR / f"{EXPERIMENT}_last.pth"


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 与当前 X0/X1/X2 正式脚本保持一致
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 82)
print("WHU formal comparison:", EXPERIMENT)
print("=" * 82)
print("Device :", DEVICE)
print("PyTorch:", torch.__version__)

if DEVICE != "cuda":
    raise RuntimeError(
        "当前没有启用 GPU。请在 Kaggle Notebook Settings -> Accelerator 中选择 GPU。"
    )

print("GPU    :", torch.cuda.get_device_name(0))


# ============================================================
# 1. 自动寻找 WHU train / val / test
# ============================================================
def locate_whu():
    if MANUAL_WHU_ROOT:
        root = Path(MANUAL_WHU_ROOT)

        paths = (
            root / "train" / "image",
            root / "train" / "label",
            root / "val" / "image",
            root / "val" / "label",
            root / "test" / "image",
            root / "test" / "label",
        )

        if all(p.is_dir() for p in paths):
            return paths

        raise FileNotFoundError(
            f"MANUAL_WHU_ROOT 下未找到标准 train/val/test 目录：{root}"
        )

    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]

    candidates = []

    for top in search_roots:
        if not top.exists():
            continue

        for train_dir in top.rglob("train"):
            root = train_dir.parent

            paths = (
                root / "train" / "image",
                root / "train" / "label",
                root / "val" / "image",
                root / "val" / "label",
                root / "test" / "image",
                root / "test" / "label",
            )

            if all(p.is_dir() for p in paths):
                candidates.append(paths)

    if not candidates:
        raise FileNotFoundError(
            "没有自动找到 WHU 数据。\n"
            "请把 MANUAL_WHU_ROOT 改成包含 train / val / test 的 whu_data 根目录。"
        )

    candidates.sort(
        key=lambda x: (
            "whu" not in str(x[0]).lower(),
            len(str(x[0]))
        )
    )

    return candidates[0]


(
    TRAIN_IMG,
    TRAIN_LBL,
    VAL_IMG,
    VAL_LBL,
    TEST_IMG,
    TEST_LBL,
) = locate_whu()

print("\nWHU data:")
print("TRAIN_IMG:", TRAIN_IMG)
print("TRAIN_LBL:", TRAIN_LBL)
print("VAL_IMG  :", VAL_IMG)
print("VAL_LBL  :", VAL_LBL)
print("TEST_IMG :", TEST_IMG)
print("TEST_LBL :", TEST_LBL)


# ============================================================
# 2. Dataset
#    与正式 X0/X1/X2 完全相同的图像预处理：
#    Resize(256) + ToTensor，不额外 Normalize
# ============================================================
class WHUDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, img_size=256):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.img_size = img_size

        valid_ext = {
            ".png", ".jpg", ".jpeg",
            ".tif", ".tiff"
        }

        self.names = sorted(
            p.name
            for p in self.img_dir.iterdir()
            if p.is_file()
            and p.suffix.lower() in valid_ext
        )

        if not self.names:
            raise RuntimeError(
                f"未找到图像：{self.img_dir}"
            )

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]

        img_path = self.img_dir / name
        lbl_path = self.lbl_dir / name

        if not lbl_path.exists():
            raise FileNotFoundError(
                f"缺少同名标签：{lbl_path}"
            )

        image = Image.open(
            img_path
        ).convert("RGB")

        label = Image.open(
            lbl_path
        ).convert("L")

        image = image.resize(
            (self.img_size, self.img_size),
            Image.BILINEAR
        )

        label = label.resize(
            (self.img_size, self.img_size),
            Image.NEAREST
        )

        img_t = T.ToTensor()(image)

        lbl_np = (
            np.array(label) > 127
        ).astype(np.uint8)

        return (
            img_t,
            torch.from_numpy(lbl_np).long(),
            name
        )


train_ds = WHUDataset(
    TRAIN_IMG,
    TRAIN_LBL,
    IMG_SIZE
)

val_ds = WHUDataset(
    VAL_IMG,
    VAL_LBL,
    IMG_SIZE
)

test_ds = WHUDataset(
    TEST_IMG,
    TEST_LBL,
    IMG_SIZE
)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=g,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("\nSamples:")
print("Train:", len(train_ds))
print("Val  :", len(val_ds))
print("Test :", len(test_ds))

EXPECTED_COUNTS = {
    "train": 4736,
    "val": 1036,
    "test": 2416,
}

actual_counts = {
    "train": len(train_ds),
    "val": len(val_ds),
    "test": len(test_ds),
}

if actual_counts != EXPECTED_COUNTS:
    print("\nWARNING: 样本数量与当前正式论文记录不一致！")
    print("Expected:", EXPECTED_COUNTS)
    print("Actual  :", actual_counts)
    print("请确认是否选错了 WHU 数据目录。")
else:
    print("Dataset count check: OK")


# ============================================================
# 3. DeepLabV3+ 网络结构
#    ResNet18 encoder + ASPP + low-level decoder
# ============================================================
class ConvBNReLU(nn.Sequential):
    def __init__(
        self,
        in_ch,
        out_ch,
        kernel_size=3,
        padding=1,
        dilation=1,
    ):
        super().__init__(
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size=kernel_size,
                padding=padding,
                dilation=dilation,
                bias=False,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class ASPPConv(nn.Sequential):
    def __init__(
        self,
        in_ch,
        out_ch,
        dilation,
    ):
        super().__init__(
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size=3,
                padding=dilation,
                dilation=dilation,
                bias=False,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class ASPPPooling(nn.Module):
    def __init__(
        self,
        in_ch,
        out_ch,
    ):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size=1,
                bias=False,
            ),
            # 这里不用 BatchNorm：
            # batch=8 但全局池化后空间为1×1，
            # 省去训练时 BN 对极小空间的额外依赖。
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        size = x.shape[-2:]

        y = self.pool(x)
        y = self.proj(y)

        return F.interpolate(
            y,
            size=size,
            mode="bilinear",
            align_corners=False,
        )


class ASPP(nn.Module):
    def __init__(
        self,
        in_ch=512,
        out_ch=256,
        rates=(3, 6, 9),
    ):
        super().__init__()

        self.branches = nn.ModuleList(
            [
                ConvBNReLU(
                    in_ch,
                    out_ch,
                    kernel_size=1,
                    padding=0,
                ),
                ASPPConv(
                    in_ch,
                    out_ch,
                    rates[0],
                ),
                ASPPConv(
                    in_ch,
                    out_ch,
                    rates[1],
                ),
                ASPPConv(
                    in_ch,
                    out_ch,
                    rates[2],
                ),
                ASPPPooling(
                    in_ch,
                    out_ch,
                ),
            ]
        )

        self.project = nn.Sequential(
            nn.Conv2d(
                out_ch * 5,
                out_ch,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )

    def forward(self, x):
        features = [
            branch(x)
            for branch in self.branches
        ]

        x = torch.cat(
            features,
            dim=1
        )

        return self.project(x)


class DeepLabV3PlusResNet18(nn.Module):
    """
    DeepLabV3+ 风格：
    - ResNet18 layer1: low-level feature H/4, 64ch
    - ResNet18 layer4: high-level feature H/32, 512ch
    - ASPP(high-level)
    - low-level projection 64->48
    - fusion + decoder
    - 2-class logits
    """

    def __init__(
        self,
        num_classes=2,
    ):
        super().__init__()

        try:
            backbone = models.resnet18(
                weights=models.ResNet18_Weights.DEFAULT
            )
        except Exception as e:
            raise RuntimeError(
                "无法取得 ImageNet 预训练 ResNet18 权重。\n"
                "请开启 Kaggle Internet 后重新运行。\n"
                "为保证与正式 X0/X1/X2 及 U-Net 对比的初始化条件一致，"
                "脚本不会自动退回随机初始化。"
            ) from e

        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        # layer4是H/32，因此采用相对更小的rates
        self.aspp = ASPP(
            in_ch=512,
            out_ch=256,
            rates=(3, 6, 9),
        )

        self.low_level_proj = nn.Sequential(
            nn.Conv2d(
                64,
                48,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
        )

        self.decoder = nn.Sequential(
            ConvBNReLU(
                256 + 48,
                256,
                kernel_size=3,
                padding=1,
            ),
            ConvBNReLU(
                256,
                256,
                kernel_size=3,
                padding=1,
            ),
            nn.Conv2d(
                256,
                num_classes,
                kernel_size=1,
            ),
        )

    def forward(self, x):
        input_hw = x.shape[-2:]

        # stem H/2
        x = self.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        # H/4
        x = self.maxpool(x)
        low = self.layer1(x)

        # H/8 -> H/16 -> H/32
        x = self.layer2(low)
        x = self.layer3(x)
        high = self.layer4(x)

        high = self.aspp(high)

        low = self.low_level_proj(
            low
        )

        high = F.interpolate(
            high,
            size=low.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        fused = torch.cat(
            [high, low],
            dim=1
        )

        logits = self.decoder(
            fused
        )

        logits = F.interpolate(
            logits,
            size=input_hw,
            mode="bilinear",
            align_corners=False,
        )

        return logits


# ============================================================
# 4. Loss：与 X0/X1/X2 / U-Net 对比保持一致
# ============================================================
ce_loss = nn.CrossEntropyLoss()


def dice_loss_building(
    logits,
    target,
    eps=1e-6
):
    prob = torch.softmax(
        logits,
        dim=1
    )[:, 1]

    gt = (
        target == 1
    ).float()

    inter = (
        prob * gt
    ).sum(
        dim=(1, 2)
    )

    denom = (
        prob.sum(dim=(1, 2))
        + gt.sum(dim=(1, 2))
    )

    dice = (
        2 * inter + eps
    ) / (
        denom + eps
    )

    return 1 - dice.mean()


# ============================================================
# 5. 统一评价指标
# ============================================================
def get_boundary(
    mask,
    kernel_size=3
):
    m = (
        mask.astype(np.uint8)
        * 255
    )

    k = np.ones(
        (kernel_size, kernel_size),
        np.uint8
    )

    dil = cv2.dilate(
        m,
        k,
        iterations=1
    )

    ero = cv2.erode(
        m,
        k,
        iterations=1
    )

    return (
        (dil - ero) > 0
    ).astype(np.uint8)


def boundary_metrics(
    gt,
    pred,
    tolerance=2
):
    gb = get_boundary(
        gt,
        3
    )

    pb = get_boundary(
        pred,
        3
    )

    # strict Boundary IoU
    inter = np.logical_and(
        gb,
        pb
    ).sum()

    union = np.logical_or(
        gb,
        pb
    ).sum()

    biou = (
        inter
        / (union + 1e-8)
    )

    # tolerance-based Boundary F1
    k = np.ones(
        (
            2 * tolerance + 1,
            2 * tolerance + 1
        ),
        np.uint8
    )

    gb_d = cv2.dilate(
        gb,
        k,
        iterations=1
    )

    pb_d = cv2.dilate(
        pb,
        k,
        iterations=1
    )

    matched_pred = np.logical_and(
        pb == 1,
        gb_d == 1
    ).sum()

    matched_gt = np.logical_and(
        gb == 1,
        pb_d == 1
    ).sum()

    bp = (
        matched_pred
        / ((pb == 1).sum() + 1e-8)
    )

    br = (
        matched_gt
        / ((gb == 1).sum() + 1e-8)
    )

    bf1 = (
        2 * bp * br
        / (bp + br + 1e-8)
    )

    return bf1, biou


@torch.no_grad()
def evaluate(
    model,
    loader
):
    model.eval()

    tp = fp = fn = tn = 0

    bf1_list = []
    biou_list = []

    for (
        images,
        labels,
        names
    ) in loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)

        pred = torch.argmax(
            logits,
            dim=1
        )

        p = (
            pred.cpu()
            .numpy()
            .astype(np.uint8)
        )

        g = (
            labels.cpu()
            .numpy()
            .astype(np.uint8)
        )

        tp += np.logical_and(
            p == 1,
            g == 1
        ).sum()

        fp += np.logical_and(
            p == 1,
            g == 0
        ).sum()

        fn += np.logical_and(
            p == 0,
            g == 1
        ).sum()

        tn += np.logical_and(
            p == 0,
            g == 0
        ).sum()

        for pi, gi in zip(p, g):
            bf1, biou = boundary_metrics(
                gi,
                pi,
                tolerance=2
            )

            bf1_list.append(
                bf1
            )

            biou_list.append(
                biou
            )

    building_iou = (
        tp
        / (tp + fp + fn + 1e-8)
    )

    bg_iou = (
        tn
        / (tn + fp + fn + 1e-8)
    )

    true_miou = (
        building_iou + bg_iou
    ) / 2

    precision = (
        tp
        / (tp + fp + 1e-8)
    )

    recall = (
        tp
        / (tp + fn + 1e-8)
    )

    f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return {
        "BuildingIoU":
            float(building_iou),

        "True_mIoU":
            float(true_miou),

        "Precision":
            float(precision),

        "Recall":
            float(recall),

        "F1":
            float(f1),

        "BoundaryF1":
            float(
                np.mean(
                    bf1_list
                )
            ),

        "BoundaryIoU":
            float(
                np.mean(
                    biou_list
                )
            ),
    }


def count_params_m(model):
    return (
        sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
        / 1e6
    )


# ============================================================
# 6. 建立模型 / optimizer / AMP
# ============================================================
seed_everything(SEED)

model = DeepLabV3PlusResNet18(
    num_classes=2
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

params_m = count_params_m(
    model
)

print("\nModel:")
print(
    "DeepLabV3+ (ResNet18 encoder)"
)
print(
    "Params_M:",
    f"{params_m:.6f}"
)


# ============================================================
# 7. 可选断点续跑
# ============================================================
start_epoch = 1
best_iou = -1.0
history = []

if (
    AUTO_RESUME
    and LAST_PATH.exists()
):
    print(
        "\n发现 last checkpoint，尝试继续训练："
    )
    print(
        LAST_PATH
    )

    last = torch.load(
        LAST_PATH,
        map_location=DEVICE
    )

    model.load_state_dict(
        last[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        last[
            "optimizer_state_dict"
        ]
    )

    if (
        "scaler_state_dict" in last
        and last[
            "scaler_state_dict"
        ] is not None
    ):
        scaler.load_state_dict(
            last[
                "scaler_state_dict"
            ]
        )

    best_iou = float(
        last.get(
            "best_iou",
            -1.0
        )
    )

    history = list(
        last.get(
            "history",
            []
        )
    )

    start_epoch = int(
        last.get(
            "epoch",
            0
        )
    ) + 1

    print(
        f"Resume from epoch {start_epoch}"
    )
    print(
        "Current best val Building IoU:",
        f"{best_iou*100:.4f}%"
    )


# ============================================================
# 8. 正式训练
# ============================================================
for epoch in range(
    start_epoch,
    EPOCHS + 1
):
    model.train()

    total_loss = 0.0
    batches = 0

    t0 = time.time()

    for (
        images,
        labels,
        names
    ) in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=True
        ):
            logits = model(
                images
            )

            loss = (
                ce_loss(
                    logits,
                    labels
                )
                + dice_loss_building(
                    logits,
                    labels
                )
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        total_loss += float(
            loss.item()
        )

        batches += 1

    avg_loss = (
        total_loss
        / max(
            batches,
            1
        )
    )

    val_metrics = evaluate(
        model,
        val_loader
    )

    row = {
        "epoch":
            epoch,

        "train_loss":
            avg_loss,

        **val_metrics
    }

    history.append(
        row
    )

    if (
        val_metrics[
            "BuildingIoU"
        ]
        > best_iou
    ):
        best_iou = (
            val_metrics[
                "BuildingIoU"
            ]
        )

        torch.save(
            model.state_dict(),
            BEST_PATH
        )

    torch.save(
        {
            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scaler_state_dict":
                scaler.state_dict(),

            "best_iou":
                best_iou,

            "history":
                history,

            "experiment":
                EXPERIMENT,
        },
        LAST_PATH
    )

    pd.DataFrame(
        history
    ).to_csv(
        RESULT_DIR
        / f"{EXPERIMENT}_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss={avg_loss:.4f} | "
        f"ValIoU={val_metrics['BuildingIoU']*100:.2f}% | "
        f"ValF1={val_metrics['F1']*100:.2f}% | "
        f"ValBF1={val_metrics['BoundaryF1']*100:.2f}% | "
        f"BestIoU={best_iou*100:.2f}% | "
        f"{time.time()-t0:.1f}s"
    )


# ============================================================
# 9. 加载最佳 checkpoint
# ============================================================
if not BEST_PATH.exists():
    raise FileNotFoundError(
        f"没有找到最佳权重：{BEST_PATH}"
    )

best_model = DeepLabV3PlusResNet18(
    num_classes=2
).to(DEVICE)

best_model.load_state_dict(
    torch.load(
        BEST_PATH,
        map_location=DEVICE
    )
)

best_model.eval()


# ============================================================
# 10. best模型：Validation统一评估
# ============================================================
val_best = evaluate(
    best_model,
    val_loader
)

val_best_pct = {
    k: v * 100.0
    for k, v
    in val_best.items()
}


# ============================================================
# 11. 锁定 best 后进行一次 independent TEST
# ============================================================
test_metrics = evaluate(
    best_model,
    test_loader
)

test_metrics_pct = {
    k: v * 100.0
    for k, v
    in test_metrics.items()
}


# ============================================================
# 12. 保存配置 / JSON
# ============================================================
config = {
    "experiment":
        EXPERIMENT,

    "paper_method_name":
        "DeepLabV3+ (ResNet18 encoder)",

    "comparison_role":
        "formal WHU comparison baseline",

    "seed":
        SEED,

    "img_size":
        IMG_SIZE,

    "batch_size":
        BATCH_SIZE,

    "epochs":
        EPOCHS,

    "optimizer":
        "Adam",

    "lr":
        LR,

    "main_loss":
        "CrossEntropy + Dice",

    "edge_loss":
        "disabled",

    "best_checkpoint_rule":
        "maximum validation Building IoU",

    "prediction_rule":
        "argmax over 2-class logits",

    "postprocessing":
        "none",

    "encoder":
        "ImageNet-pretrained ResNet18",

    "architecture":
        (
            "ResNet18 encoder + ASPP(rates=3,6,9) "
            "+ layer1 low-level fusion + DeepLabV3+ decoder"
        ),

    "high_level_output_stride":
        32,

    "input_normalization":
        (
            "ToTensor only; no additional ImageNet normalization, "
            "matching X0/X1/X2 formal protocol"
        ),

    "boundary_definition":
        "3x3 morphological boundary",

    "boundary_f1_tolerance_px":
        2,

    "boundary_iou":
        "strict intersection-over-union",

    "params_M":
        count_params_m(
            best_model
        ),

    "train_samples":
        len(train_ds),

    "val_samples":
        len(val_ds),

    "test_samples":
        len(test_ds),

    "train_img":
        str(TRAIN_IMG),

    "val_img":
        str(VAL_IMG),

    "test_img":
        str(TEST_IMG),

    "device":
        torch.cuda.get_device_name(0),
}

(
    RESULT_DIR
    / f"{EXPERIMENT}_config.json"
).write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

(
    RESULT_DIR
    / f"{EXPERIMENT}_val_metrics.json"
).write_text(
    json.dumps(
        val_best_pct,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

(
    RESULT_DIR
    / f"{EXPERIMENT}_test_metrics.json"
).write_text(
    json.dumps(
        test_metrics_pct,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 13. TXT报告
# ============================================================
report_lines = [
    (
        "WHU formal comparison experiment: "
        + EXPERIMENT
    ),
    "=" * 82,
    "",
    "Experiment config",
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    ),
    "",
    "Best checkpoint VALIDATION metrics (%)",
]

report_lines.extend(
    f"{k}: {v:.4f}"
    for k, v
    in val_best_pct.items()
)

report_lines.extend(
    [
        "",
        "Independent TEST metrics (%)",
    ]
)

report_lines.extend(
    f"{k}: {v:.4f}"
    for k, v
    in test_metrics_pct.items()
)

report = "\n".join(
    report_lines
)

REPORT_PATH = (
    RESULT_DIR
    / f"{EXPERIMENT}_report.txt"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8"
)

print(
    "\n" + report
)


# ============================================================
# 14. 论文记录用 summary CSV
# ============================================================
summary_df = pd.DataFrame(
    [
        {
            "Method":
                "DeepLabV3+ (ResNet18 encoder)",
            "Split":
                "Validation",
            **val_best_pct,
            "Params_M":
                count_params_m(
                    best_model
                ),
        },
        {
            "Method":
                "DeepLabV3+ (ResNet18 encoder)",
            "Split":
                "Independent Test",
            **test_metrics_pct,
            "Params_M":
                count_params_m(
                    best_model
                ),
        },
    ]
)

SUMMARY_PATH = (
    RESULT_DIR
    / f"{EXPERIMENT}_summary.csv"
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. 训练曲线
# ============================================================
hist = pd.DataFrame(
    history
)

plt.figure(
    figsize=(7.2, 4.6)
)

plt.plot(
    hist["epoch"],
    hist["train_loss"],
    linewidth=1.8
)

plt.xlabel(
    "Epoch"
)
plt.ylabel(
    "Training Loss"
)
plt.title(
    "DeepLabV3+ (ResNet18) Training Loss"
)
plt.grid(
    alpha=0.2
)
plt.tight_layout()

plt.savefig(
    FIG_DIR
    / f"{EXPERIMENT}_training_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


plt.figure(
    figsize=(7.2, 4.6)
)

plt.plot(
    hist["epoch"],
    hist["BuildingIoU"] * 100,
    linewidth=1.8,
    label="Building IoU"
)

plt.plot(
    hist["epoch"],
    hist["F1"] * 100,
    linewidth=1.8,
    label="F1"
)

plt.xlabel(
    "Epoch"
)
plt.ylabel(
    "Validation score (%)"
)
plt.title(
    "DeepLabV3+ Validation Performance"
)
plt.grid(
    alpha=0.2
)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIG_DIR
    / f"{EXPERIMENT}_validation_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 16. 打包全部结果
# ============================================================
ARCHIVE_PATH = (
    Path("/kaggle/working")
    / f"{EXPERIMENT}_artifacts.zip"
)

with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:

    for file in OUT_ROOT.rglob("*"):
        if file.is_file():
            zf.write(
                file,
                file.relative_to(
                    OUT_ROOT.parent
                )
            )


print(
    "\n" + "=" * 82
)
print(
    "正式 DeepLabV3+ 对比实验完成"
)
print(
    "=" * 82
)
print(
    "Best checkpoint :",
    BEST_PATH
)
print(
    "Report          :",
    REPORT_PATH
)
print(
    "Summary CSV     :",
    SUMMARY_PATH
)
print(
    "Artifacts ZIP   :",
    ARCHIVE_PATH
)

print(
    "\n论文表4建议使用 TEST 行："
)

for k, v in test_metrics_pct.items():
    print(
        f"{k}: {v:.4f}"
    )


WHU formal comparison: DeepLabV3Plus_ResNet18_Formal_Comparison
Device : cuda
PyTorch: 2.10.0+cu128
GPU    : Tesla T4

WHU data:
TRAIN_IMG: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
TRAIN_LBL: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/label
VAL_IMG  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
VAL_LBL  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/label
TEST_IMG : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/test/image
TEST_LBL : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/test/label

Samples:
Train: 4736
Val  : 1036
Test : 2416
Dataset count check: OK
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 175MB/s] 



Model:
DeepLabV3+ (ResNet18 encoder)
Params_M: 16.602786
Epoch 01/50 | Loss=0.4909 | ValIoU=74.89% | ValF1=85.64% | ValBF1=58.18% | BestIoU=74.89% | 66.9s
Epoch 02/50 | Loss=0.3861 | ValIoU=76.31% | ValF1=86.57% | ValBF1=58.44% | BestIoU=76.31% | 36.0s
Epoch 03/50 | Loss=0.3558 | ValIoU=76.12% | ValF1=86.44% | ValBF1=55.82% | BestIoU=76.31% | 36.8s
Epoch 04/50 | Loss=0.3460 | ValIoU=77.82% | ValF1=87.53% | ValBF1=59.35% | BestIoU=77.82% | 36.0s
Epoch 05/50 | Loss=0.3310 | ValIoU=74.79% | ValF1=85.58% | ValBF1=53.78% | BestIoU=77.82% | 36.3s
Epoch 06/50 | Loss=0.3213 | ValIoU=81.84% | ValF1=90.01% | ValBF1=60.39% | BestIoU=81.84% | 36.4s
Epoch 07/50 | Loss=0.3242 | ValIoU=82.45% | ValF1=90.38% | ValBF1=60.08% | BestIoU=82.45% | 36.2s
Epoch 08/50 | Loss=0.3253 | ValIoU=82.20% | ValF1=90.23% | ValBF1=60.63% | BestIoU=82.45% | 37.1s
Epoch 09/50 | Loss=0.3010 | ValIoU=83.57% | ValF1=91.05% | ValBF1=61.14% | BestIoU=83.57% | 36.4s
Epoch 10/50 | Loss=0.2986 | ValIoU=80.77% | ValF1=89.36% | V

In [1]:
import os
import sys
import subprocess
import random
import json
import time
import zipfile
from pathlib import Path

# Set before importing cv2, torch, or matplotlib. These are harmless on
# Kaggle Linux and prevent a common duplicate OpenMP runtime crash locally.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

RESAMPLING = getattr(Image, "Resampling", Image)

try:
    cv2.setNumThreads(0)
except Exception:
    pass

try:
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass

try:
    from transformers import SegformerForSemanticSegmentation
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "transformers"]
    )
    from transformers import SegformerForSemanticSegmentation

# ===== 0. 正式固定协议 =====
EXPERIMENT = "SegFormerB0_Formal_Comparison"
SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-3
NUM_WORKERS = 2
AUTO_RESUME = True

# Usually leave both values unchanged and enable Kaggle Internet. If Internet
# is disabled, attach a downloaded Hugging Face nvidia/mit-b0 model directory
# as a Kaggle dataset, then set LOCAL_MIT_B0_DIR to that directory.
MODEL_SOURCE = "nvidia/mit-b0"
LOCAL_MIT_B0_DIR = ""

# Automatic search is preferred. Only set this when the attached dataset uses
# a nonstandard directory nesting, for example:
# "/kaggle/input/whu-data/whu_data"
MANUAL_WHU_ROOT = ""

OUT = Path("/kaggle/working") / f"WHU_{EXPERIMENT}_final"
CKPT = OUT / "checkpoints"
RES = OUT / "results"
FIG = OUT / "figures"
for p in (CKPT, RES, FIG):
    p.mkdir(parents=True, exist_ok=True)

BEST = CKPT / f"{EXPERIMENT}_best.pth"
LAST = CKPT / f"{EXPERIMENT}_last.pth"

def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError(
        "当前没有启用 GPU。请在 Kaggle Notebook 的 Settings -> Accelerator "
        "中选择 GPU 后重新运行。"
    )
print("=" * 78)
print("WHU formal comparison:", EXPERIMENT)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# ===== 1. 数据路径 =====
def locate_whu():
    def build_paths(root):
        root = Path(root)
        return (
            root / "train" / "image",
            root / "train" / "label",
            root / "val" / "image",
            root / "val" / "label",
            root / "test" / "image",
            root / "test" / "label",
        )

    if MANUAL_WHU_ROOT.strip():
        paths = build_paths(MANUAL_WHU_ROOT.strip())
        if all(path.is_dir() for path in paths):
            return paths
        raise FileNotFoundError(
            f"MANUAL_WHU_ROOT 目录不完整：{MANUAL_WHU_ROOT}"
        )

    candidates = []
    seen = set()
    for top in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not top.exists():
            continue
        for train_dir in top.rglob("train"):
            if not train_dir.is_dir():
                continue
            paths = build_paths(train_dir.parent)
            if all(path.is_dir() for path in paths):
                key = tuple(str(path.resolve()) for path in paths)
                if key not in seen:
                    candidates.append(paths)
                    seen.add(key)
    if not candidates:
        raise FileNotFoundError(
            "未找到 WHU train/val/test 数据。请确认已添加 Kaggle 数据集，"
            "或填写 MANUAL_WHU_ROOT。"
        )
    candidates.sort(
        key=lambda paths: (
            "whu" not in str(paths[0]).lower(),
            len(str(paths[0])),
        )
    )
    return candidates[0]

TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL, TEST_IMG, TEST_LBL = locate_whu()
print("TRAIN:", TRAIN_IMG)
print("VAL  :", VAL_IMG)
print("TEST :", TEST_IMG)

# ===== 2. Dataset：与X0/X1/X2一致 =====
class WHU(Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        valid_extensions = {
            ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"
        }
        self.names = sorted(
            path.name
            for path in self.image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in valid_extensions
        )
        if not self.names:
            raise RuntimeError(f"未在此目录找到图像：{self.image_dir}")

        missing = [
            name for name in self.names
            if not (self.label_dir / name).is_file()
        ]
        if missing:
            raise FileNotFoundError(
                f"标签目录缺少 {len(missing)} 个同名标签。示例：{missing[:5]}"
            )

        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image = Image.open(self.image_dir / name).convert("RGB")
        label = Image.open(self.label_dir / name).convert("L")
        image = image.resize(
            (IMG_SIZE, IMG_SIZE),
            RESAMPLING.BILINEAR,
        )
        label = label.resize(
            (IMG_SIZE, IMG_SIZE),
            RESAMPLING.NEAREST,
        )
        image_tensor = self.to_tensor(image)
        label_tensor = torch.from_numpy(
            (np.asarray(label) > 127).astype(np.uint8)
        ).long()
        return image_tensor, label_tensor, name


tr = WHU(TRAIN_IMG, TRAIN_LBL)
va = WHU(VAL_IMG, VAL_LBL)
te = WHU(TEST_IMG, TEST_LBL)

generator = torch.Generator()
generator.manual_seed(SEED)
loader_args = {
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
}
if NUM_WORKERS > 0:
    loader_args["persistent_workers"] = True

trL = DataLoader(
    tr,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    **loader_args,
)
vaL = DataLoader(
    va,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_args,
)
teL = DataLoader(
    te,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_args,
)

counts = {"train": len(tr), "val": len(va), "test": len(te)}
print("Samples:", counts)
if counts != {"train": 4736, "val": 1036, "test": 2416}:
    print("WARNING: 样本数与正式论文记录不一致，请核对数据目录。")
else:
    print("Dataset count check: OK")

# ===== 3. SegFormer-B0 =====
class SegFormerB0(nn.Module):
    def __init__(self):
        super().__init__()
        source = LOCAL_MIT_B0_DIR.strip() or MODEL_SOURCE
        local_files_only = bool(LOCAL_MIT_B0_DIR.strip())
        try:
            self.net = SegformerForSemanticSegmentation.from_pretrained(
                source,
                num_labels=2,
                ignore_mismatched_sizes=True,
                id2label={0: "background", 1: "building"},
                label2id={"background": 0, "building": 1},
                local_files_only=local_files_only,
            )
        except Exception as error:
            location = (
                f"本地模型目录 {LOCAL_MIT_B0_DIR}"
                if local_files_only
                else f"Hugging Face 模型 {MODEL_SOURCE}"
            )
            raise RuntimeError(
                f"无法加载 {location}。\n"
                "请打开 Kaggle Internet，或把 Hugging Face 的 nvidia/mit-b0 "
                "完整模型目录作为 Kaggle Dataset 加入后，填写 LOCAL_MIT_B0_DIR。"
            ) from error

    def forward(self, x):
        logits = self.net(pixel_values=x).logits
        return F.interpolate(
            logits,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

# ===== 4. Loss =====
CE=nn.CrossEntropyLoss()
def dice_loss(logits,target,eps=1e-6):
    p=torch.softmax(logits,1)[:,1]; g=(target==1).float()
    inter=(p*g).sum((1,2)); den=p.sum((1,2))+g.sum((1,2))
    return 1-((2*inter+eps)/(den+eps)).mean()

# ===== 5. 统一指标 =====
def boundary(mask,k=3):
    m=mask.astype(np.uint8)*255; ker=np.ones((k,k),np.uint8)
    return ((cv2.dilate(m,ker,iterations=1)-cv2.erode(m,ker,iterations=1))>0).astype(np.uint8)

def bmetrics(gt,pred,tol=2):
    gb,pb=boundary(gt),boundary(pred)
    biou=np.logical_and(gb,pb).sum()/(np.logical_or(gb,pb).sum()+1e-8)
    ker=np.ones((2*tol+1,2*tol+1),np.uint8)
    gd=cv2.dilate(gb,ker,iterations=1); pd=cv2.dilate(pb,ker,iterations=1)
    mp=np.logical_and(pb==1,gd==1).sum(); mg=np.logical_and(gb==1,pd==1).sum()
    bp=mp/((pb==1).sum()+1e-8); br=mg/((gb==1).sum()+1e-8)
    return 2*bp*br/(bp+br+1e-8),biou

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); tp=fp=fn=tn=0; bfs=[]; bis=[]
    for x,y,n in loader:
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        p=torch.argmax(model(x),1).cpu().numpy().astype(np.uint8)
        g=y.cpu().numpy().astype(np.uint8)
        tp+=np.logical_and(p==1,g==1).sum(); fp+=np.logical_and(p==1,g==0).sum()
        fn+=np.logical_and(p==0,g==1).sum(); tn+=np.logical_and(p==0,g==0).sum()
        for pi,gi in zip(p,g):
            a,b=bmetrics(gi,pi,2); bfs.append(a); bis.append(b)
    iou=tp/(tp+fp+fn+1e-8); bgiou=tn/(tn+fp+fn+1e-8)
    pr=tp/(tp+fp+1e-8); re=tp/(tp+fn+1e-8); f1=2*pr*re/(pr+re+1e-8)
    return {"BuildingIoU":float(iou),"True_mIoU":float((iou+bgiou)/2),
            "Precision":float(pr),"Recall":float(re),"F1":float(f1),
            "BoundaryF1":float(np.mean(bfs)),"BoundaryIoU":float(np.mean(bis))}

def params_m(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)/1e6

# ===== 6. 训练 =====
seed_all(SEED)
model = SegFormerB0().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

try:
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    def autocast_context():
        return torch.amp.autocast("cuda", enabled=True)

except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=True)

    def autocast_context():
        return torch.cuda.amp.autocast(enabled=True)


print("Params_M:", f"{params_m(model):.6f}")

start = 1
best_iou = -1.0
hist = []
if AUTO_RESUME and LAST.exists():
    ck = torch.load(LAST, map_location=DEVICE)
    model.load_state_dict(ck["model_state_dict"], strict=True)
    opt.load_state_dict(ck["optimizer_state_dict"])
    if ck.get("scaler_state_dict") is not None:
        scaler.load_state_dict(ck["scaler_state_dict"])
    best_iou = float(ck.get("best_iou", -1.0))
    hist = list(ck.get("history", []))
    start = int(ck.get("epoch", 0)) + 1
    print("Resume from epoch", start)

for ep in range(start, EPOCHS + 1):
    model.train()
    total = 0.0
    nb = 0
    t0 = time.time()
    for x, y, _ in trL:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast_context():
            logits = model(x)
            loss = CE(logits, y) + dice_loss(logits, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        total += float(loss.item())
        nb += 1

    avg = total / max(nb, 1)
    vm = evaluate(model, vaL)
    hist.append({"epoch": ep, "train_loss": avg, **vm})
    if vm["BuildingIoU"] > best_iou:
        best_iou = vm["BuildingIoU"]
        torch.save(model.state_dict(), BEST)

    torch.save(
        {
            "epoch": ep,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": opt.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_iou": best_iou,
            "history": hist,
            "experiment": EXPERIMENT,
        },
        LAST,
    )
    pd.DataFrame(hist).to_csv(
        RES / f"{EXPERIMENT}_history.csv",
        index=False,
        encoding="utf-8-sig",
    )
    print(
        f"Epoch {ep:02d}/{EPOCHS} | Loss={avg:.4f} | "
        f"ValIoU={vm['BuildingIoU'] * 100:.2f}% | "
        f"ValF1={vm['F1'] * 100:.2f}% | "
        f"ValBF1={vm['BoundaryF1'] * 100:.2f}% | "
        f"BestIoU={best_iou * 100:.2f}% | {time.time() - t0:.1f}s"
    )

# ===== 7. 锁定best后验证+独立test =====
if not BEST.exists():
    raise FileNotFoundError(f"没有找到最佳权重：{BEST}")

# Reuse the existing model so the final evaluation never makes a second
# Hugging Face download request.
best = model
best.load_state_dict(torch.load(BEST, map_location=DEVICE), strict=True)
best.eval()
val_pct = {k: v * 100 for k, v in evaluate(best, vaL).items()}
test_pct = {k: v * 100 for k, v in evaluate(best, teL).items()}

config={"experiment":EXPERIMENT,"paper_method_name":"SegFormer-B0","seed":SEED,"img_size":IMG_SIZE,
        "batch_size":BATCH_SIZE,"epochs":EPOCHS,"optimizer":"Adam","lr":LR,
        "main_loss":"CrossEntropy + Dice","edge_loss":"disabled",
        "best_checkpoint_rule":"maximum validation Building IoU","prediction_rule":"argmax over 2-class logits",
        "postprocessing":"none","encoder":"ImageNet-pretrained MiT-B0 (nvidia/mit-b0)",
        "input_normalization":"ToTensor only; no extra ImageNet normalization, matching X0/X1/X2",
        "boundary_definition":"3x3 morphological boundary","boundary_f1_tolerance_px":2,
        "boundary_iou":"strict intersection-over-union","params_M":params_m(best),
        "train_samples":len(tr),"val_samples":len(va),"test_samples":len(te),
        "device":torch.cuda.get_device_name(0)}

(RES/f"{EXPERIMENT}_config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")
(RES/f"{EXPERIMENT}_val_metrics.json").write_text(json.dumps(val_pct,ensure_ascii=False,indent=2),encoding="utf-8")
(RES/f"{EXPERIMENT}_test_metrics.json").write_text(json.dumps(test_pct,ensure_ascii=False,indent=2),encoding="utf-8")

lines=[f"WHU formal comparison experiment: {EXPERIMENT}","="*82,"","Experiment config",
       json.dumps(config,ensure_ascii=False,indent=2),"","Best checkpoint VALIDATION metrics (%)"]
lines += [f"{k}: {v:.4f}" for k,v in val_pct.items()]
lines += ["","Independent TEST metrics (%)"]+[f"{k}: {v:.4f}" for k,v in test_pct.items()]
report="\n".join(lines)
REPORT=RES/f"{EXPERIMENT}_report.txt"; REPORT.write_text(report,encoding="utf-8")
print("\n"+report)

pd.DataFrame([
    {"Method":"SegFormer-B0","Split":"Validation",**val_pct,"Params_M":params_m(best)},
    {"Method":"SegFormer-B0","Split":"Independent Test",**test_pct,"Params_M":params_m(best)}
]).to_csv(RES/f"{EXPERIMENT}_summary.csv",index=False,encoding="utf-8-sig")

h=pd.DataFrame(hist)
plt.figure(figsize=(7.2,4.6)); plt.plot(h["epoch"],h["train_loss"]); plt.xlabel("Epoch"); plt.ylabel("Training Loss")
plt.grid(alpha=.2); plt.tight_layout(); plt.savefig(FIG/f"{EXPERIMENT}_training_loss.png",dpi=300,bbox_inches="tight"); plt.close()

ARCHIVE=Path("/kaggle/working")/f"{EXPERIMENT}_artifacts.zip"
with zipfile.ZipFile(ARCHIVE,"w",zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file(): z.write(f,f.relative_to(OUT.parent))

print("\n完成。Artifacts ZIP:",ARCHIVE)
print("论文表4使用 TEST 行：")
for k,v in test_pct.items(): print(f"{k}: {v:.4f}")

WHU formal comparison: SegFormerB0_Formal_Comparison
PyTorch: 2.10.0+cu128
GPU: Tesla T4
TRAIN: /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/train/image
VAL  : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/val/image
TEST : /kaggle/input/datasets/xiaomaoperson/whu-data/whu_data/test/image


Samples: {'train': 4736, 'val': 1036, 'test': 2416}
Dataset count check: OK


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

model.safetensors:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

Params_M: 3.714658
Epoch 01/50 | Loss=0.5150 | ValIoU=67.41% | ValF1=80.53% | ValBF1=51.65% | BestIoU=67.41% | 71.4s
Epoch 02/50 | Loss=0.4174 | ValIoU=71.14% | ValF1=83.13% | ValBF1=52.79% | BestIoU=71.14% | 41.1s
Epoch 03/50 | Loss=0.3887 | ValIoU=79.37% | ValF1=88.50% | ValBF1=58.50% | BestIoU=79.37% | 41.7s
Epoch 04/50 | Loss=0.3611 | ValIoU=76.81% | ValF1=86.89% | ValBF1=57.91% | BestIoU=79.37% | 41.3s
Epoch 05/50 | Loss=0.3662 | ValIoU=79.87% | ValF1=88.81% | ValBF1=58.72% | BestIoU=79.87% | 41.1s
Epoch 06/50 | Loss=0.3477 | ValIoU=79.83% | ValF1=88.79% | ValBF1=59.08% | BestIoU=79.87% | 41.1s
Epoch 07/50 | Loss=0.3532 | ValIoU=81.28% | ValF1=89.67% | ValBF1=59.73% | BestIoU=81.28% | 41.3s
Epoch 08/50 | Loss=0.3384 | ValIoU=80.66% | ValF1=89.29% | ValBF1=60.18% | BestIoU=81.28% | 41.5s
Epoch 09/50 | Loss=0.3756 | ValIoU=77.49% | ValF1=87.32% | ValBF1=54.28% | BestIoU=81.28% | 41.2s
Epoch 10/50 | Loss=0.4311 | ValIoU=73.24% | ValF1=84.56% | ValBF1=53.51% | BestIoU=81.28% | 41.1s
E